In [1]:
import pandas as pd
from google.colab import files

# 파일 업로드
uploaded = files.upload()

filename = list(uploaded.keys())[0]

# Dong-Month Behavior Mart 불러오기
df_month = pd.read_csv(filename, encoding="utf-8-sig")

print("Shape:", df_month.shape)
print("행정동 수:", df_month["행정동코드"].nunique())
print("월 수:", df_month["기준연월"].nunique())
print("중복 Key:",
      df_month.duplicated(
          subset=["행정동코드", "기준연월"]
      ).sum())
print("전체 결측치:", df_month.isna().sum().sum())

print("\n=== 전체 컬럼 ===")
for i, col in enumerate(df_month.columns, 1):
    print(f"{i:02d}. {col}")

Saving gangnam_db1_dong_month_behavior_mart_2025_07_12.csv to gangnam_db1_dong_month_behavior_mart_2025_07_12.csv
Shape: (132, 39)
행정동 수: 22
월 수: 6
중복 Key: 0
전체 결측치: 0

=== 전체 컬럼 ===
01. 행정동코드
02. 행정동명
03. 기준연월
04. total_flow
05. flow_per_point
06. male_ratio
07. age_20_ratio
08. age_30_ratio
09. age_40_ratio
10. age_50_ratio
11. age_60_plus_ratio
12. morning_ratio
13. daytime_ratio
14. evening_ratio
15. peak_hour_ratio
16. weekend_weekday_index
17. peak_day_ratio
18. low_comm_ratio
19. low_weekday_outing_ratio
20. low_holiday_outing_ratio
21. high_worktime_ratio
22. high_video_ratio
23. high_life_service_ratio
24. financial_interest_ratio
25. very_low_outing_ratio
26. very_low_outing_comm_ratio
27. 야간상주지 변경횟수 평균
28. 평균 근무시간 평균
29. 평일 총 이동 횟수
30. 휴일 총 이동 횟수 평균
31. 집 추정 위치 평일 총 체류시간
32. 집 추정 위치 휴일 총 체류시간
33. 금융 서비스 사용일수
34. 쇼핑 서비스 사용일수
35. 배달 서비스 사용일수
36. 최근 3개월 내 요금 연체 비율
37. 평균 통화대상자 수
38. 데이터 사용량
39. 카카오톡_비사용비율


In [2]:
import pandas as pd
import numpy as np

# 식별자
id_cols = ["행정동코드", "행정동명", "기준연월"]

# 36개 분석 Feature
feature_cols = [c for c in df_month.columns if c not in id_cols]

results = []

for col in feature_cols:

    # -------------------------
    # 1. 지역 간 차이
    # 각 동의 6개월 평균 → 22개 동 평균값의 표준편차
    # -------------------------
    dong_mean = (
        df_month
        .groupby("행정동코드")[col]
        .mean()
    )

    between_std = dong_mean.std(ddof=1)

    # -------------------------
    # 2. 동 내부 월별 변화
    # 각 동의 6개월 표준편차 → 22개 동 평균
    # -------------------------
    dong_within_std = (
        df_month
        .groupby("행정동코드")[col]
        .std(ddof=1)
    )

    within_std = dong_within_std.mean()

    # -------------------------
    # 3. 동 내부 변화 / 지역 간 차이
    # -------------------------
    if pd.isna(between_std) or between_std == 0:
        ratio = np.nan
    else:
        ratio = within_std / between_std

    # -------------------------
    # 4. 전체 평균 대비 동 내부 변화 정도
    # 변수 단위가 서로 다르므로 참고용 CV
    # -------------------------
    overall_mean = df_month[col].mean()

    if pd.isna(overall_mean) or overall_mean == 0:
        within_cv = np.nan
    else:
        within_cv = within_std / abs(overall_mean)

    results.append({
        "feature": col,
        "overall_mean": overall_mean,
        "between_std": between_std,
        "within_std": within_std,
        "within_between_ratio": ratio,
        "within_cv": within_cv
    })

variability_df = pd.DataFrame(results)

# 동 내부 변화 / 지역 간 차이가 큰 순서
variability_df = (
    variability_df
    .sort_values(
        "within_between_ratio",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Feature 수:", len(variability_df))
print("\n=== 동별 자기기준 변화 가능성 진단 ===")

print(
    variability_df[
        [
            "feature",
            "between_std",
            "within_std",
            "within_between_ratio",
            "within_cv"
        ]
    ].to_string(index=False)
)

Feature 수: 36

=== 동별 자기기준 변화 가능성 진단 ===
                   feature   between_std    within_std  within_between_ratio  within_cv
             야간상주지 변경횟수 평균      0.065137      1.722460             26.443723   0.159447
         집 추정 위치 평일 총 체류시간  25044.311287 191786.940131              7.657904   0.261541
             휴일 총 이동 횟수 평균      0.562083      2.244881              3.993860   0.194575
                평일 총 이동 횟수      1.887470      6.397880              3.389659   0.246453
         집 추정 위치 휴일 총 체류시간  10614.599409  31342.090397              2.952734   0.211757
                   데이터 사용량      1.044600      1.607544              1.538909   0.220487
                평균 근무시간 평균      8.304362      7.841742              0.944292   0.014949
               쇼핑 서비스 사용일수     10.604247      9.872321              0.930978   0.040082
               금융 서비스 사용일수     11.492856      7.146861              0.621852   0.040644
very_low_outing_comm_ratio      0.341238      0.208885              0.612138   

In [3]:
import pandas as pd
import numpy as np

core_behavior = [
    "평일 총 이동 횟수",
    "휴일 총 이동 횟수 평균",
    "집 추정 위치 평일 총 체류시간",
    "집 추정 위치 휴일 총 체류시간"
]

# 월 순서 보장
df_check = df_month.copy()
df_check["기준연월"] = df_check["기준연월"].astype(str)
df_check = df_check.sort_values(["행정동코드", "기준연월"])

# ------------------------------------------------
# 1. 22개 동의 월별 평균
# ------------------------------------------------
monthly_mean = (
    df_check
    .groupby("기준연월")[core_behavior]
    .mean()
)

print("=== 1. 22개 동 월별 평균 ===")
print(monthly_mean.round(3).to_string())


# ------------------------------------------------
# 2. 월별 평균의 전월 대비 변화율(%)
# ------------------------------------------------
monthly_pct_change = monthly_mean.pct_change() * 100

print("\n=== 2. 월별 평균 전월 대비 변화율(%) ===")
print(monthly_pct_change.round(2).to_string())


# ------------------------------------------------
# 3. 각 동별 전월 대비 증감 계산
# ------------------------------------------------
for col in core_behavior:
    df_check[f"{col}_diff"] = (
        df_check
        .groupby("행정동코드")[col]
        .diff()
    )


# ------------------------------------------------
# 4. 각 월에 몇 개 동이 증가/감소했는지 확인
# ------------------------------------------------
direction_results = []

for month in sorted(df_check["기준연월"].unique())[1:]:

    month_df = df_check[df_check["기준연월"] == month]

    for col in core_behavior:
        diff_col = f"{col}_diff"

        valid = month_df[diff_col].dropna()

        increase = (valid > 0).sum()
        decrease = (valid < 0).sum()
        same = (valid == 0).sum()
        total = len(valid)

        dominant_count = max(increase, decrease, same)

        dominant_ratio = (
            dominant_count / total
            if total > 0 else np.nan
        )

        if increase > decrease and increase > same:
            dominant_direction = "증가"
        elif decrease > increase and decrease > same:
            dominant_direction = "감소"
        else:
            dominant_direction = "혼합"

        direction_results.append({
            "기준연월": month,
            "feature": col,
            "증가_동수": increase,
            "감소_동수": decrease,
            "변화없음_동수": same,
            "우세방향": dominant_direction,
            "우세방향비율": dominant_ratio
        })

direction_df = pd.DataFrame(direction_results)

print("\n=== 3. 전월 대비 변화 방향 일치도 ===")
print(
    direction_df.to_string(
        index=False,
        formatters={
            "우세방향비율": lambda x: f"{x:.1%}"
        }
    )
)


# ------------------------------------------------
# 5. 공통 월 효과가 강한 경우만 별도 출력
# 임시 진단기준: 80% 이상 같은 방향
# ------------------------------------------------
strong_common = direction_df[
    direction_df["우세방향비율"] >= 0.80
].copy()

print("\n=== 4. 80% 이상 동이 같은 방향으로 움직인 경우 ===")

if len(strong_common) == 0:
    print("없음")
else:
    print(
        strong_common.to_string(
            index=False,
            formatters={
                "우세방향비율": lambda x: f"{x:.1%}"
            }
        )
    )

=== 1. 22개 동 월별 평균 ===
         평일 총 이동 횟수  휴일 총 이동 횟수 평균  집 추정 위치 평일 총 체류시간  집 추정 위치 휴일 총 체류시간
기준연월                                                                    
2025-07      21.193         11.202         578937.705         122802.887
2025-08      23.055          9.235         646063.768         117570.466
2025-09      35.392         13.141        1015856.396         167384.378
2025-10      32.800         15.218         937728.015         200728.328
2025-11      22.238          9.947         622534.366         139463.640
2025-12      21.080         10.481         598660.531         140109.204

=== 2. 월별 평균 전월 대비 변화율(%) ===
         평일 총 이동 횟수  휴일 총 이동 횟수 평균  집 추정 위치 평일 총 체류시간  집 추정 위치 휴일 총 체류시간
기준연월                                                                    
2025-07         NaN            NaN                NaN                NaN
2025-08        8.78         -17.56              11.59              -4.26
2025-09       53.51          42.28              57.24              42.

In [4]:
import pandas as pd
import numpy as np

id_cols = ["행정동코드", "행정동명", "기준연월"]
feature_cols = [c for c in df_month.columns if c not in id_cols]

df_common = df_month.copy()
df_common["기준연월"] = df_common["기준연월"].astype(str)
df_common = df_common.sort_values(["행정동코드", "기준연월"])

# ---------------------------------------------
# 1. Feature별 동 단위 전월 차이 계산
# ---------------------------------------------
for col in feature_cols:
    df_common[f"{col}__diff"] = (
        df_common
        .groupby("행정동코드")[col]
        .diff()
    )

# ---------------------------------------------
# 2. 월 × Feature별 변화방향 일치도
# ---------------------------------------------
monthly_direction_results = []

months = sorted(df_common["기준연월"].unique())

for month in months[1:]:

    temp = df_common[df_common["기준연월"] == month]

    for col in feature_cols:

        diff = temp[f"{col}__diff"].dropna()

        increase = int((diff > 0).sum())
        decrease = int((diff < 0).sum())
        same = int((diff == 0).sum())
        total = len(diff)

        counts = {
            "증가": increase,
            "감소": decrease,
            "변화없음": same
        }

        dominant_direction = max(counts, key=counts.get)
        dominant_count = counts[dominant_direction]

        dominant_ratio = (
            dominant_count / total
            if total > 0 else np.nan
        )

        monthly_direction_results.append({
            "기준연월": month,
            "feature": col,
            "증가_동수": increase,
            "감소_동수": decrease,
            "변화없음_동수": same,
            "우세방향": dominant_direction,
            "방향일치율": dominant_ratio
        })

monthly_direction_df = pd.DataFrame(monthly_direction_results)

# ---------------------------------------------
# 3. Feature별 공통 월 효과 요약
# ---------------------------------------------
common_summary = (
    monthly_direction_df
    .groupby("feature")
    .agg(
        평균_방향일치율=("방향일치율", "mean"),
        최소_방향일치율=("방향일치율", "min"),
        최대_방향일치율=("방향일치율", "max"),
        강한공통효과_80이상_횟수=(
            "방향일치율",
            lambda x: int((x >= 0.80).sum())
        ),
        완전일치_100_횟수=(
            "방향일치율",
            lambda x: int((x == 1.0).sum())
        )
    )
    .reset_index()
)

# 앞서 계산한 변동성 결과도 결합
common_summary = common_summary.merge(
    variability_df[
        [
            "feature",
            "within_between_ratio",
            "within_cv"
        ]
    ],
    on="feature",
    how="left"
)

common_summary = (
    common_summary
    .sort_values(
        [
            "평균_방향일치율",
            "강한공통효과_80이상_횟수"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

# ---------------------------------------------
# 4. 전체 요약 출력
# ---------------------------------------------
display_summary = common_summary.copy()

for col in [
    "평균_방향일치율",
    "최소_방향일치율",
    "최대_방향일치율"
]:
    display_summary[col] = (
        display_summary[col] * 100
    ).round(1)

display_summary["within_between_ratio"] = (
    display_summary["within_between_ratio"].round(3)
)

display_summary["within_cv"] = (
    display_summary["within_cv"].round(3)
)

print("=== 36개 Feature 공통 월 효과 진단 ===")
print(display_summary.to_string(index=False))


# ---------------------------------------------
# 5. 특히 강한 공통효과 Feature
# 임시 진단:
# 5번 중 3번 이상에서 80% 이상 같은 방향
# ---------------------------------------------
strong_features = common_summary[
    common_summary["강한공통효과_80이상_횟수"] >= 3
].copy()

print("\n=== 강한 공통 월 효과 후보 ===")
print("Feature 수:", len(strong_features))

if len(strong_features) == 0:
    print("없음")
else:
    strong_display = strong_features.copy()

    strong_display["평균_방향일치율"] = (
        strong_display["평균_방향일치율"] * 100
    ).round(1)

    print(
        strong_display[
            [
                "feature",
                "평균_방향일치율",
                "강한공통효과_80이상_횟수",
                "완전일치_100_횟수",
                "within_between_ratio",
                "within_cv"
            ]
        ].to_string(index=False)
    )

=== 36개 Feature 공통 월 효과 진단 ===
                   feature  평균_방향일치율  최소_방향일치율  최대_방향일치율  강한공통효과_80이상_횟수  완전일치_100_횟수  within_between_ratio  within_cv
               금융 서비스 사용일수     100.0     100.0     100.0               5            5                 0.622      0.041
             야간상주지 변경횟수 평균     100.0     100.0     100.0               5            5                26.444      0.159
                평일 총 이동 횟수     100.0     100.0     100.0               5            5                 3.390      0.246
             휴일 총 이동 횟수 평균     100.0     100.0     100.0               5            5                 3.994      0.195
             morning_ratio      99.1      95.5     100.0               5            4                 0.220      0.022
         집 추정 위치 평일 총 체류시간      99.1      95.5     100.0               5            4                 7.658      0.262
                평균 통화대상자 수      97.3      86.4     100.0               5            4                 0.203      0.012
                 

In [5]:
import pandas as pd
import numpy as np

a2_features = [
    "평일 총 이동 횟수",
    "휴일 총 이동 횟수 평균",
    "집 추정 위치 평일 총 체류시간",
    "집 추정 위치 휴일 총 체류시간",
    "평균 통화대상자 수"
]

df_method = df_month.copy()
df_method["기준연월"] = df_method["기준연월"].astype(str)
df_method = df_method.sort_values(["행정동코드", "기준연월"])

# -------------------------------------------------
# 1. 원자료 분포 확인
# -------------------------------------------------
distribution_results = []

for col in a2_features:

    s = df_method[col].dropna()

    distribution_results.append({
        "feature": col,
        "min": s.min(),
        "p25": s.quantile(0.25),
        "median": s.median(),
        "mean": s.mean(),
        "p75": s.quantile(0.75),
        "max": s.max(),
        "std": s.std(),
        "skew": s.skew(),
        "zero_count": int((s == 0).sum())
    })

distribution_df = pd.DataFrame(distribution_results)

print("=== 1. A2 원자료 분포 ===")
print(distribution_df.round(3).to_string(index=False))


# -------------------------------------------------
# 2. 동별 전월 변화 계산
# -------------------------------------------------
change_results = []

for col in a2_features:

    grouped = df_method.groupby("행정동코드")[col]

    previous = grouped.shift(1)

    # 단순 차이
    absolute_change = df_method[col] - previous

    # 변화율
    pct_change = (
        (df_method[col] - previous)
        / previous.replace(0, np.nan)
    )

    # 로그 변화율
    # 모든 값이 양수일 경우 log(x_t / x_t-1)
    log_change = np.log(
        df_method[col] / previous.replace(0, np.nan)
    )

    temp = pd.DataFrame({
        "행정동코드": df_method["행정동코드"],
        "행정동명": df_method["행정동명"],
        "기준연월": df_method["기준연월"],
        "feature": col,
        "absolute_change": absolute_change,
        "pct_change": pct_change,
        "log_change": log_change
    })

    change_results.append(temp)

change_df = pd.concat(change_results, ignore_index=True)


# -------------------------------------------------
# 3. 변화량 분포 비교
# -------------------------------------------------
method_results = []

for col in a2_features:

    temp = change_df[
        change_df["feature"] == col
    ]

    for method in [
        "absolute_change",
        "pct_change",
        "log_change"
    ]:

        s = temp[method].replace(
            [np.inf, -np.inf],
            np.nan
        ).dropna()

        method_results.append({
            "feature": col,
            "method": method,
            "mean": s.mean(),
            "median": s.median(),
            "std": s.std(),
            "min": s.min(),
            "max": s.max(),
            "skew": s.skew()
        })

method_compare_df = pd.DataFrame(method_results)

print("\n=== 2. 전월 변화 계산방법별 분포 ===")
print(
    method_compare_df
    .round(4)
    .to_string(index=False)
)


# -------------------------------------------------
# 4. 월별 로그 변화의 동 간 분포
# 공통 월 효과를 제거할 때 중앙값 사용 가능성 확인
# -------------------------------------------------
log_monthly_results = []

for col in a2_features:

    temp = change_df[
        change_df["feature"] == col
    ]

    for month in sorted(
        temp["기준연월"].dropna().unique()
    )[1:]:

        s = (
            temp.loc[
                temp["기준연월"] == month,
                "log_change"
            ]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        log_monthly_results.append({
            "feature": col,
            "기준연월": month,
            "mean_log_change": s.mean(),
            "median_log_change": s.median(),
            "std_log_change": s.std(),
            "min_log_change": s.min(),
            "max_log_change": s.max()
        })

log_monthly_df = pd.DataFrame(log_monthly_results)

print("\n=== 3. 월별 로그 변화 분포 ===")
print(
    log_monthly_df
    .round(4)
    .to_string(index=False)
)

=== 1. A2 원자료 분포 ===
          feature        min        p25     median       mean        p75         max        std   skew  zero_count
       평일 총 이동 횟수     18.312     21.518     23.246     25.960     31.961      39.588      6.165  0.748           0
    휴일 총 이동 횟수 평균      8.353      9.911     10.780     11.537     13.042      16.557      2.133  0.724           0
집 추정 위치 평일 총 체류시간 531095.888 599313.953 637008.010 733296.797 939003.705 1069065.121 177660.264  0.725           0
집 추정 위치 휴일 총 체류시간  99773.332 124349.848 141889.611 148009.817 168282.201  224904.808  30631.672  0.721           0
       평균 통화대상자 수     11.385     13.184     13.835     13.604     14.156      15.073      0.811 -0.896           0

=== 2. 전월 변화 계산방법별 분포 ===
          feature          method      mean      median         std          min         max    skew
       평일 총 이동 횟수 absolute_change   -0.0226     -1.1000      7.4776     -11.8786     13.6765  0.3521
       평일 총 이동 횟수      pct_change    0.0352     -0.0509     

In [6]:
import pandas as pd
import numpy as np

a2_features = [
    "평일 총 이동 횟수",
    "휴일 총 이동 횟수 평균",
    "집 추정 위치 평일 총 체류시간",
    "집 추정 위치 휴일 총 체류시간",
    "평균 통화대상자 수"
]

df_a2 = df_month[
    ["행정동코드", "행정동명", "기준연월"] + a2_features
].copy()

df_a2["기준연월"] = df_a2["기준연월"].astype(str)

df_a2 = df_a2.sort_values(
    ["행정동코드", "기준연월"]
).reset_index(drop=True)


# =====================================================
# 1. 동별 전월 대비 로그 변화
# =====================================================

for col in a2_features:

    previous = (
        df_a2
        .groupby("행정동코드")[col]
        .shift(1)
    )

    df_a2[f"{col}__log_change"] = np.log(
        df_a2[col] / previous
    )


# =====================================================
# 2. 월별 공통효과 = 22개 동 로그변화의 중앙값
# =====================================================

for col in a2_features:

    log_col = f"{col}__log_change"

    df_a2[f"{col}__month_median"] = (
        df_a2
        .groupby("기준연월")[log_col]
        .transform("median")
    )


# =====================================================
# 3. 공통 월 효과 제거
# residual = 동의 로그변화 - 월별 중앙값
# =====================================================

for col in a2_features:

    df_a2[f"{col}__residual"] = (
        df_a2[f"{col}__log_change"]
        - df_a2[f"{col}__month_median"]
    )


# =====================================================
# 4. 월별 Robust Z-score
#
# Robust Z = residual / (1.4826 × MAD)
#
# residual의 월별 중앙값은 구조상 0에 가까우므로
# MAD = median(|residual - median(residual)|)
# =====================================================

mad_diagnostics = []

for col in a2_features:

    residual_col = f"{col}__residual"
    z_col = f"{col}__robust_z"

    df_a2[z_col] = np.nan

    for month in sorted(df_a2["기준연월"].unique()):

        mask = df_a2["기준연월"] == month

        s = (
            df_a2.loc[mask, residual_col]
            .dropna()
        )

        if len(s) == 0:
            continue

        median_residual = s.median()

        mad = np.median(
            np.abs(s - median_residual)
        )

        robust_scale = 1.4826 * mad

        mad_diagnostics.append({
            "feature": col,
            "기준연월": month,
            "median_residual": median_residual,
            "MAD": mad,
            "robust_scale": robust_scale
        })

        # MAD가 0인 경우에는 Z-score 계산하지 않음
        if robust_scale > 0:

            df_a2.loc[mask, z_col] = (
                df_a2.loc[mask, residual_col]
                - median_residual
            ) / robust_scale


mad_df = pd.DataFrame(mad_diagnostics)


# =====================================================
# 5. 7월 제외 → 실제 변화 분석 테이블
# =====================================================

a2_change = df_a2[
    df_a2["기준연월"] != "2025-07"
].copy()


# =====================================================
# 6. 검증 ① 기본 구조
# =====================================================

z_cols = [
    f"{col}__robust_z"
    for col in a2_features
]

print("=== 1. A2 변화 테이블 기본 검증 ===")
print("shape:", a2_change.shape)
print(
    "행정동 수:",
    a2_change["행정동코드"].nunique()
)
print(
    "월 수:",
    a2_change["기준연월"].nunique()
)
print(
    "동×월 중복:",
    a2_change.duplicated(
        ["행정동코드", "기준연월"]
    ).sum()
)
print(
    "Robust Z 결측 수:",
    a2_change[z_cols].isna().sum().sum()
)


# =====================================================
# 7. 검증 ② MAD / Robust Scale
# =====================================================

print("\n=== 2. 월별 MAD 진단 ===")

print(
    mad_df[
        mad_df["기준연월"] != "2025-07"
    ]
    .round(6)
    .to_string(index=False)
)


# =====================================================
# 8. 검증 ③ Robust Z 분포
# =====================================================

z_summary = []

for col in a2_features:

    z_col = f"{col}__robust_z"

    s = a2_change[z_col].dropna()

    z_summary.append({
        "feature": col,
        "min_z": s.min(),
        "p25_z": s.quantile(0.25),
        "median_z": s.median(),
        "p75_z": s.quantile(0.75),
        "max_z": s.max(),
        "|z|>=2": int((s.abs() >= 2).sum()),
        "|z|>=3": int((s.abs() >= 3).sum())
    })

z_summary_df = pd.DataFrame(z_summary)

print("\n=== 3. Robust Z 전체 분포 ===")
print(
    z_summary_df
    .round(3)
    .to_string(index=False)
)


# =====================================================
# 9. 검증 ④ 월별 Robust Z 중앙값
# 정상이라면 거의 0
# =====================================================

monthly_z_median = (
    a2_change
    .groupby("기준연월")[z_cols]
    .median()
)

print("\n=== 4. 월별 Robust Z 중앙값 ===")
print(
    monthly_z_median
    .round(6)
    .to_string()
)


# =====================================================
# 10. 절대값 기준 가장 큰 관측치 확인
# 아직 '이상신호' 판정은 아님
# =====================================================

extreme_rows = []

for col in a2_features:

    z_col = f"{col}__robust_z"

    temp = a2_change[
        [
            "행정동코드",
            "행정동명",
            "기준연월",
            col,
            f"{col}__log_change",
            f"{col}__month_median",
            f"{col}__residual",
            z_col
        ]
    ].copy()

    temp["feature"] = col
    temp["abs_z"] = temp[z_col].abs()

    temp = (
        temp
        .sort_values("abs_z", ascending=False)
        .head(3)
    )

    extreme_rows.append(temp)

extreme_df = pd.concat(
    extreme_rows,
    ignore_index=True
)

print("\n=== 5. Feature별 |Robust Z| 상위 3개 ===")

for col in a2_features:

    print(f"\n[{col}]")

    temp = extreme_df[
        extreme_df["feature"] == col
    ]

    print(
        temp[
            [
                "행정동명",
                "기준연월",
                f"{col}__log_change",
                f"{col}__month_median",
                f"{col}__residual",
                f"{col}__robust_z"
            ]
        ]
        .round(4)
        .to_string(index=False)
    )

=== 1. A2 변화 테이블 기본 검증 ===
shape: (110, 28)
행정동 수: 22
월 수: 5
동×월 중복: 0
Robust Z 결측 수: 0

=== 2. 월별 MAD 진단 ===
          feature    기준연월  median_residual      MAD  robust_scale
       평일 총 이동 횟수 2025-08              0.0 0.002931      0.004346
       평일 총 이동 횟수 2025-09             -0.0 0.004886      0.007244
       평일 총 이동 횟수 2025-10              0.0 0.002609      0.003869
       평일 총 이동 횟수 2025-11              0.0 0.003685      0.005463
       평일 총 이동 횟수 2025-12             -0.0 0.006074      0.009005
    휴일 총 이동 횟수 평균 2025-08              0.0 0.005118      0.007588
    휴일 총 이동 횟수 평균 2025-09              0.0 0.005281      0.007830
    휴일 총 이동 횟수 평균 2025-10              0.0 0.003494      0.005180
    휴일 총 이동 횟수 평균 2025-11              0.0 0.006427      0.009528
    휴일 총 이동 횟수 평균 2025-12              0.0 0.005458      0.008092
집 추정 위치 평일 총 체류시간 2025-08             -0.0 0.008928      0.013237
집 추정 위치 평일 총 체류시간 2025-09             -0.0 0.010379      0.015387
집 추정 위치 평일 총 체류시간 2025-10       

In [7]:
import pandas as pd
import numpy as np

# =====================================================
# 1. 방향 통일
# +값 = 사회적 고립 관련 행동방향과 정합적인 상대변화
# =====================================================

signal_map = {
    "weekday_move_signal":
        -a2_change["평일 총 이동 횟수__robust_z"],

    "holiday_move_signal":
        -a2_change["휴일 총 이동 횟수 평균__robust_z"],

    "weekday_home_signal":
        a2_change["집 추정 위치 평일 총 체류시간__robust_z"],

    "holiday_home_signal":
        a2_change["집 추정 위치 휴일 총 체류시간__robust_z"],

    "call_partner_signal":
        -a2_change["평균 통화대상자 수__robust_z"]
}

a2_signal = a2_change[
    ["행정동코드", "행정동명", "기준연월"]
].copy()

for name, values in signal_map.items():
    a2_signal[name] = values.values


signal_cols = list(signal_map.keys())


# =====================================================
# 2. 기본 검증
# =====================================================

print("=== 1. 방향 통일 A2 Signal 기본 검증 ===")
print("shape:", a2_signal.shape)
print(
    "동×월 중복:",
    a2_signal.duplicated(
        ["행정동코드", "기준연월"]
    ).sum()
)
print(
    "결측:",
    a2_signal[signal_cols].isna().sum().sum()
)


# =====================================================
# 3. Signal 간 상관관계
# 같은 방향의 변화가 실제로 함께 발생하는지 확인
# =====================================================

signal_corr = a2_signal[signal_cols].corr()

print("\n=== 2. A2 Signal 상관관계 ===")
print(
    signal_corr
    .round(3)
    .to_string()
)


# =====================================================
# 4. 동×월별 양(+)의 방향 신호 개수
#
# 여기서는 임계값을 두지 않음.
# 단순히 0보다 큰 방향이 몇 개인지만 진단.
# =====================================================

a2_signal["positive_direction_count"] = (
    a2_signal[signal_cols] > 0
).sum(axis=1)


count_dist = (
    a2_signal["positive_direction_count"]
    .value_counts()
    .sort_index()
)

print("\n=== 3. 동×월별 양(+) 방향 Signal 개수 분포 ===")
print(count_dist.to_string())


# =====================================================
# 5. 5개 중 4개 이상이 같은 방향인 동×월 확인
#
# 아직 '위험' 또는 '이상' 판정이 아님
# =====================================================

co_direction = (
    a2_signal[
        a2_signal["positive_direction_count"] >= 4
    ]
    .sort_values(
        ["positive_direction_count", "기준연월"],
        ascending=[False, True]
    )
)

print("\n=== 4. 5개 중 4개 이상이 관찰방향과 일치한 동×월 ===")

if len(co_direction) == 0:
    print("없음")
else:
    print(
        co_direction[
            [
                "행정동명",
                "기준연월",
                "positive_direction_count"
            ] + signal_cols
        ]
        .round(3)
        .to_string(index=False)
    )


# =====================================================
# 6. 각 Signal에서 양의 방향이 강했던 관측치
# 임시 진단용으로 Z >= 2만 출력
# 이것을 이상신호 기준으로 확정하지 않음
# =====================================================

strong_rows = []

for col in signal_cols:

    temp = a2_signal[
        a2_signal[col] >= 2
    ][
        [
            "행정동명",
            "기준연월",
            col
        ]
    ].copy()

    temp["signal"] = col
    temp = temp.rename(
        columns={col: "signal_value"}
    )

    strong_rows.append(temp)

strong_signal_df = pd.concat(
    strong_rows,
    ignore_index=True
)

print("\n=== 5. 참고: 방향 통일 Signal >= 2 관측치 ===")

if len(strong_signal_df) == 0:
    print("없음")
else:
    print(
        strong_signal_df
        .sort_values(
            "signal_value",
            ascending=False
        )
        .round(3)
        .to_string(index=False)
    )

=== 1. 방향 통일 A2 Signal 기본 검증 ===
shape: (110, 8)
동×월 중복: 0
결측: 0

=== 2. A2 Signal 상관관계 ===
                     weekday_move_signal  holiday_move_signal  weekday_home_signal  holiday_home_signal  call_partner_signal
weekday_move_signal                1.000                0.740                0.607                0.396                0.176
holiday_move_signal                0.740                1.000                0.464                0.276                0.081
weekday_home_signal                0.607                0.464                1.000                0.628                0.078
holiday_home_signal                0.396                0.276                0.628                1.000               -0.036
call_partner_signal                0.176                0.081                0.078               -0.036                1.000

=== 3. 동×월별 양(+) 방향 Signal 개수 분포 ===
positive_direction_count
0    13
1    21
2    19
3    24
4    21
5    12

=== 4. 5개 중 4개 이상이 관찰방향과 일치한 동×월 ===
행정동명    기

In [8]:
import pandas as pd
import numpy as np

a2_axis = a2_signal[
    ["행정동코드", "행정동명", "기준연월"]
].copy()

# =====================================================
# 1. A2 3개 행동축 생성
# =====================================================

# 외부활동 감소 방향
a2_axis["outside_activity_axis"] = (
    a2_signal["weekday_move_signal"]
    + a2_signal["holiday_move_signal"]
) / 2

# 집 체류 증가 방향
a2_axis["home_stay_axis"] = (
    a2_signal["weekday_home_signal"]
    + a2_signal["holiday_home_signal"]
) / 2

# 대인소통 감소 방향
a2_axis["communication_axis"] = (
    a2_signal["call_partner_signal"]
)

axis_cols = [
    "outside_activity_axis",
    "home_stay_axis",
    "communication_axis"
]


# =====================================================
# 2. 기본 검증
# =====================================================

print("=== 1. A2 3축 기본 검증 ===")
print("shape:", a2_axis.shape)
print(
    "동×월 중복:",
    a2_axis.duplicated(
        ["행정동코드", "기준연월"]
    ).sum()
)
print(
    "결측:",
    a2_axis[axis_cols].isna().sum().sum()
)


# =====================================================
# 3. 3축 분포
# =====================================================

axis_summary = []

for col in axis_cols:

    s = a2_axis[col]

    axis_summary.append({
        "axis": col,
        "min": s.min(),
        "p25": s.quantile(0.25),
        "median": s.median(),
        "mean": s.mean(),
        "p75": s.quantile(0.75),
        "max": s.max(),
        "std": s.std()
    })

axis_summary_df = pd.DataFrame(axis_summary)

print("\n=== 2. A2 3축 분포 ===")
print(
    axis_summary_df
    .round(3)
    .to_string(index=False)
)


# =====================================================
# 4. 3축 상관관계
# =====================================================

print("\n=== 3. A2 3축 상관관계 ===")

print(
    a2_axis[axis_cols]
    .corr()
    .round(3)
    .to_string()
)


# =====================================================
# 5. 세 축 방향의 동시성
#
# 아직 임계값을 적용하지 않고
# 단순히 + 방향인지 여부만 확인
# =====================================================

a2_axis["positive_axis_count"] = (
    a2_axis[axis_cols] > 0
).sum(axis=1)

print("\n=== 4. 양(+) 방향 행동축 개수 분포 ===")

print(
    a2_axis["positive_axis_count"]
    .value_counts()
    .sort_index()
    .to_string()
)


# =====================================================
# 6. 세 행동축이 모두 +인 동×월
# 아직 위험/이상 판정 아님
# =====================================================

all_positive = (
    a2_axis[
        a2_axis["positive_axis_count"] == 3
    ]
    .sort_values(
        ["기준연월", "행정동명"]
    )
)

print("\n=== 5. 세 행동축이 모두 +인 동×월 ===")
print("관측치 수:", len(all_positive))

if len(all_positive) == 0:
    print("없음")
else:
    print(
        all_positive[
            [
                "행정동명",
                "기준연월",
                "outside_activity_axis",
                "home_stay_axis",
                "communication_axis"
            ]
        ]
        .round(3)
        .to_string(index=False)
    )


# =====================================================
# 7. 세 축의 단순 평균
#
# 최종 점수가 아님.
# 분포 확인을 위한 진단 변수만 생성.
# =====================================================

a2_axis["three_axis_mean"] = (
    a2_axis[axis_cols].mean(axis=1)
)

print("\n=== 6. 3축 단순 평균 분포 - 진단용 ===")

print(
    a2_axis["three_axis_mean"]
    .describe()
    .round(3)
    .to_string()
)

print("\n=== 7. 3축 단순 평균 상위 10개 - 진단용 ===")

print(
    a2_axis[
        [
            "행정동명",
            "기준연월",
            "outside_activity_axis",
            "home_stay_axis",
            "communication_axis",
            "three_axis_mean"
        ]
    ]
    .sort_values(
        "three_axis_mean",
        ascending=False
    )
    .head(10)
    .round(3)
    .to_string(index=False)
)

=== 1. A2 3축 기본 검증 ===
shape: (110, 6)
동×월 중복: 0
결측: 0

=== 2. A2 3축 분포 ===
                 axis    min    p25  median  mean   p75   max   std
outside_activity_axis -2.591 -0.604  -0.034 0.018 0.645 2.617 1.019
       home_stay_axis -2.400 -0.439  -0.103 0.146 0.778 2.851 1.003
   communication_axis -3.205 -0.691   0.000 0.054 0.653 2.859 1.044

=== 3. A2 3축 상관관계 ===
                       outside_activity_axis  home_stay_axis  communication_axis
outside_activity_axis                  1.000           0.538               0.141
home_stay_axis                         0.538           1.000               0.033
communication_axis                     0.141           0.033               1.000

=== 4. 양(+) 방향 행동축 개수 분포 ===
positive_axis_count
0    22
1    36
2    34
3    18

=== 5. 세 행동축이 모두 +인 동×월 ===
관측치 수: 18
행정동명    기준연월  outside_activity_axis  home_stay_axis  communication_axis
개포2동 2025-08                  0.624           0.362               0.508
개포3동 2025-08                  1.154     

In [9]:
import pandas as pd
import numpy as np

a1_features = [
    "low_comm_ratio",
    "low_weekday_outing_ratio",
    "low_holiday_outing_ratio",
    "very_low_outing_ratio",
    "very_low_outing_comm_ratio"
]

df_a1 = df_month[
    ["행정동코드", "행정동명", "기준연월"] + a1_features
].copy()

df_a1["기준연월"] = df_a1["기준연월"].astype(str)

df_a1 = df_a1.sort_values(
    ["행정동코드", "기준연월"]
).reset_index(drop=True)


# =====================================================
# 1. 원자료 상관관계
# =====================================================

print("=== 1. A1 원자료 상관관계 ===")

print(
    df_a1[a1_features]
    .corr()
    .round(3)
    .to_string()
)


# =====================================================
# 2. 행정동별 전월 절대 변화량 계산
# 관심집단 비율이므로 우선 percentage-point 변화 확인
# =====================================================

change_cols = []

for col in a1_features:

    change_col = f"{col}__change"

    df_a1[change_col] = (
        df_a1
        .groupby("행정동코드")[col]
        .diff()
    )

    change_cols.append(change_col)


# 7월 제외
a1_change = df_a1[
    df_a1["기준연월"] != "2025-07"
].copy()


# =====================================================
# 3. 전월 변화량 분포
# =====================================================

change_summary = []

for col in a1_features:

    change_col = f"{col}__change"
    s = a1_change[change_col].dropna()

    change_summary.append({
        "feature": col,
        "min_change": s.min(),
        "p25_change": s.quantile(0.25),
        "median_change": s.median(),
        "mean_change": s.mean(),
        "p75_change": s.quantile(0.75),
        "max_change": s.max(),
        "std_change": s.std(),
        "mean_abs_change": s.abs().mean()
    })

change_summary_df = pd.DataFrame(change_summary)

print("\n=== 2. A1 전월 변화량 분포 ===")

print(
    change_summary_df
    .round(4)
    .to_string(index=False)
)


# =====================================================
# 4. 전월 변화량끼리의 상관관계
# 원자료 상관보다 이번 분석에서는 이것이 더 중요
# =====================================================

print("\n=== 3. A1 전월 변화량 상관관계 ===")

print(
    a1_change[change_cols]
    .corr()
    .round(3)
    .to_string()
)


# =====================================================
# 5. 월별 변화량 중앙값과 동 간 분산
# =====================================================

monthly_change = (
    a1_change
    .groupby("기준연월")[change_cols]
    .agg(["median", "std"])
)

print("\n=== 4. A1 월별 변화량 중앙값 / 표준편차 ===")

print(
    monthly_change
    .round(4)
    .to_string()
)


# =====================================================
# 6. 값이 거의 변하지 않는 동×월 비율 확인
# 정확한 0 및 작은 변화량 확인
# =====================================================

zero_summary = []

for col in a1_features:

    change_col = f"{col}__change"
    s = a1_change[change_col]

    zero_summary.append({
        "feature": col,
        "exact_zero_count": int((s == 0).sum()),
        "abs_change_lt_0.01": int((s.abs() < 0.01).sum()),
        "total": int(s.notna().sum())
    })

zero_summary_df = pd.DataFrame(zero_summary)

print("\n=== 5. A1 변화 희소성 진단 ===")

print(
    zero_summary_df.to_string(index=False)
)

=== 1. A1 원자료 상관관계 ===
                            low_comm_ratio  low_weekday_outing_ratio  low_holiday_outing_ratio  very_low_outing_ratio  very_low_outing_comm_ratio
low_comm_ratio                       1.000                     0.699                     0.679                  0.489                       0.457
low_weekday_outing_ratio             0.699                     1.000                     0.235                  0.671                       0.533
low_holiday_outing_ratio             0.679                     0.235                     1.000                  0.027                       0.321
very_low_outing_ratio                0.489                     0.671                     0.027                  1.000                       0.621
very_low_outing_comm_ratio           0.457                     0.533                     0.321                  0.621                       1.000

=== 2. A1 전월 변화량 분포 ===
                   feature  min_change  p25_change  median_change  mean_chan

In [10]:
import pandas as pd
import numpy as np

a1_features = [
    "low_comm_ratio",
    "low_weekday_outing_ratio",
    "low_holiday_outing_ratio",
    "very_low_outing_ratio",
    "very_low_outing_comm_ratio"
]

# 이미 만들어둔 df_a1 사용
a1_robust = df_a1.copy()

mad_results = []

for col in a1_features:

    change_col = f"{col}__change"
    median_col = f"{col}__month_median"
    residual_col = f"{col}__residual"
    z_col = f"{col}__robust_z"

    # ---------------------------------------------
    # 1. 월별 전월 변화량 중앙값
    # ---------------------------------------------
    a1_robust[median_col] = (
        a1_robust
        .groupby("기준연월")[change_col]
        .transform("median")
    )

    # ---------------------------------------------
    # 2. 공통 월 효과 제거
    # ---------------------------------------------
    a1_robust[residual_col] = (
        a1_robust[change_col]
        - a1_robust[median_col]
    )

    a1_robust[z_col] = np.nan

    # ---------------------------------------------
    # 3. 월별 Robust Z
    # ---------------------------------------------
    for month in sorted(
        a1_robust["기준연월"].unique()
    ):

        mask = (
            a1_robust["기준연월"] == month
        )

        s = (
            a1_robust
            .loc[mask, residual_col]
            .dropna()
        )

        if len(s) == 0:
            continue

        median_residual = s.median()

        mad = np.median(
            np.abs(
                s - median_residual
            )
        )

        robust_scale = 1.4826 * mad

        mad_results.append({
            "feature": col,
            "기준연월": month,
            "median_change":
                a1_robust.loc[
                    mask,
                    change_col
                ].median(),
            "MAD": mad,
            "robust_scale": robust_scale
        })

        if robust_scale > 0:

            a1_robust.loc[
                mask,
                z_col
            ] = (
                a1_robust.loc[
                    mask,
                    residual_col
                ]
                - median_residual
            ) / robust_scale


# 7월 제외
a1_signal = a1_robust[
    a1_robust["기준연월"] != "2025-07"
].copy()

z_cols = [
    f"{col}__robust_z"
    for col in a1_features
]

mad_a1_df = pd.DataFrame(
    mad_results
)


# =====================================================
# 4. 기본 검증
# =====================================================

print("=== 1. A1 Robust 변화신호 기본 검증 ===")

print("shape:", a1_signal.shape)

print(
    "행정동 수:",
    a1_signal["행정동코드"].nunique()
)

print(
    "월 수:",
    a1_signal["기준연월"].nunique()
)

print(
    "동×월 중복:",
    a1_signal.duplicated(
        ["행정동코드", "기준연월"]
    ).sum()
)

print(
    "Robust Z 결측:",
    a1_signal[z_cols]
    .isna()
    .sum()
    .sum()
)


# =====================================================
# 5. MAD 진단
# =====================================================

print("\n=== 2. A1 월별 MAD 진단 ===")

print(
    mad_a1_df[
        mad_a1_df["기준연월"] != "2025-07"
    ]
    .round(4)
    .to_string(index=False)
)


# =====================================================
# 6. Robust Z 분포
# =====================================================

z_summary = []

for col in a1_features:

    z_col = f"{col}__robust_z"

    s = a1_signal[z_col].dropna()

    z_summary.append({
        "feature": col,
        "min_z": s.min(),
        "p25_z": s.quantile(0.25),
        "median_z": s.median(),
        "p75_z": s.quantile(0.75),
        "max_z": s.max(),
        "|z|>=2": int(
            (s.abs() >= 2).sum()
        ),
        "|z|>=3": int(
            (s.abs() >= 3).sum()
        )
    })

z_summary_df = pd.DataFrame(
    z_summary
)

print("\n=== 3. A1 Robust Z 분포 ===")

print(
    z_summary_df
    .round(3)
    .to_string(index=False)
)


# =====================================================
# 7. 월별 중앙값 검증
# =====================================================

print("\n=== 4. A1 월별 Robust Z 중앙값 ===")

print(
    a1_signal
    .groupby("기준연월")[z_cols]
    .median()
    .round(6)
    .to_string()
)


# =====================================================
# 8. 양의 방향 상위 관측치
#
# A1은 ratio 증가 자체가
# 관찰하려는 방향이므로 부호 반전 필요 없음
# =====================================================

strong_rows = []

for col in a1_features:

    z_col = f"{col}__robust_z"

    temp = (
        a1_signal[
            [
                "행정동명",
                "기준연월",
                change_col if False else z_col
            ]
        ]
        .copy()
    )

    temp["feature"] = col
    temp["signal_value"] = (
        temp[z_col]
    )

    temp = (
        temp
        .sort_values(
            "signal_value",
            ascending=False
        )
        .head(3)
    )

    strong_rows.append(
        temp[
            [
                "행정동명",
                "기준연월",
                "feature",
                "signal_value"
            ]
        ]
    )

strong_a1_df = pd.concat(
    strong_rows,
    ignore_index=True
)

print(
    "\n=== 5. A1 Feature별 양(+) Robust Z 상위 3개 ==="
)

print(
    strong_a1_df
    .sort_values(
        "signal_value",
        ascending=False
    )
    .round(3)
    .to_string(index=False)
)

=== 1. A1 Robust 변화신호 기본 검증 ===
shape: (110, 28)
행정동 수: 22
월 수: 5
동×월 중복: 0
Robust Z 결측: 0

=== 2. A1 월별 MAD 진단 ===
                   feature    기준연월  median_change    MAD  robust_scale
            low_comm_ratio 2025-08         0.0458 0.1308        0.1940
            low_comm_ratio 2025-09        -0.0893 0.1383        0.2050
            low_comm_ratio 2025-10        -0.0498 0.1707        0.2531
            low_comm_ratio 2025-11         0.0944 0.1892        0.2805
            low_comm_ratio 2025-12         0.0704 0.1724        0.2555
  low_weekday_outing_ratio 2025-08        -0.1715 0.2936        0.4353
  low_weekday_outing_ratio 2025-09         0.3137 0.3655        0.5419
  low_weekday_outing_ratio 2025-10         0.5441 0.3065        0.4544
  low_weekday_outing_ratio 2025-11         0.0346 0.4178        0.6194
  low_weekday_outing_ratio 2025-12        -0.1427 0.3727        0.5526
  low_holiday_outing_ratio 2025-08         0.2497 0.2244        0.3326
  low_holiday_outing_ratio 2025-

In [11]:
import pandas as pd
import numpy as np

# =====================================================
# 1. A1 5개 Robust Z를 간단한 이름으로 정리
# =====================================================

a1_final = a1_signal[
    ["행정동코드", "행정동명", "기준연월"]
].copy()

a1_final["low_comm_signal"] = (
    a1_signal["low_comm_ratio__robust_z"]
)

a1_final["low_weekday_outing_signal"] = (
    a1_signal["low_weekday_outing_ratio__robust_z"]
)

a1_final["low_holiday_outing_signal"] = (
    a1_signal["low_holiday_outing_ratio__robust_z"]
)

a1_final["very_low_outing_signal"] = (
    a1_signal["very_low_outing_ratio__robust_z"]
)

a1_final["very_low_outing_comm_signal"] = (
    a1_signal["very_low_outing_comm_ratio__robust_z"]
)


# =====================================================
# 2. A2 3축과 결합
# =====================================================

behavior_signal = a1_final.merge(
    a2_axis[
        [
            "행정동코드",
            "행정동명",
            "기준연월",
            "outside_activity_axis",
            "home_stay_axis",
            "communication_axis"
        ]
    ],
    on=[
        "행정동코드",
        "행정동명",
        "기준연월"
    ],
    how="inner",
    validate="one_to_one"
)


a1_cols = [
    "low_comm_signal",
    "low_weekday_outing_signal",
    "low_holiday_outing_signal",
    "very_low_outing_signal",
    "very_low_outing_comm_signal"
]

a2_cols = [
    "outside_activity_axis",
    "home_stay_axis",
    "communication_axis"
]

signal_cols = a1_cols + a2_cols


# =====================================================
# 3. 기본 검증
# =====================================================

print("=== 1. 통합 행동신호 테이블 검증 ===")

print("shape:", behavior_signal.shape)

print(
    "행정동 수:",
    behavior_signal["행정동코드"].nunique()
)

print(
    "월 수:",
    behavior_signal["기준연월"].nunique()
)

print(
    "동×월 중복:",
    behavior_signal.duplicated(
        ["행정동코드", "기준연월"]
    ).sum()
)

print(
    "결측:",
    behavior_signal[signal_cols]
    .isna()
    .sum()
    .sum()
)


# =====================================================
# 4. 8개 행동신호 상관관계
# =====================================================

print("\n=== 2. A1 + A2 행동신호 상관관계 ===")

print(
    behavior_signal[signal_cols]
    .corr()
    .round(3)
    .to_string()
)


# =====================================================
# 5. A1과 A2 사이의 상관만 별도 추출
# =====================================================

cross_corr = (
    behavior_signal[signal_cols]
    .corr()
    .loc[a1_cols, a2_cols]
)

print("\n=== 3. A1 ↔ A2 교차 상관관계 ===")

print(
    cross_corr
    .round(3)
    .to_string()
)


# =====================================================
# 6. A1 / A2 각각 양의 방향 개수
#
# 아직 임계값 없는 방향성 진단
# =====================================================

behavior_signal["a1_positive_count"] = (
    behavior_signal[a1_cols] > 0
).sum(axis=1)

behavior_signal["a2_positive_count"] = (
    behavior_signal[a2_cols] > 0
).sum(axis=1)


print("\n=== 4. A1 양(+) 개수 × A2 양(+) 개수 교차표 ===")

print(
    pd.crosstab(
        behavior_signal["a1_positive_count"],
        behavior_signal["a2_positive_count"]
    ).to_string()
)


# =====================================================
# 7. 비교적 정합적인 동×월 탐색
#
# A1 5개 중 3개 이상 +
# AND
# A2 3축 중 2개 이상 +
#
# 아직 탐지규칙이 아니라 진단 조건
# =====================================================

coherent = behavior_signal[
    (behavior_signal["a1_positive_count"] >= 3)
    &
    (behavior_signal["a2_positive_count"] >= 2)
].copy()

print("\n=== 5. A1≥3개 + / A2≥2축 + 동×월 ===")
print("관측치 수:", len(coherent))

print(
    coherent[
        [
            "행정동명",
            "기준연월",
            "a1_positive_count",
            "a2_positive_count"
        ] + signal_cols
    ]
    .sort_values(
        ["기준연월", "행정동명"]
    )
    .round(3)
    .to_string(index=False)
)


# =====================================================
# 8. 조금 더 엄격한 '강도' 동시성 진단
#
# A1에서 Z >= 1인 신호 개수
# A2에서 축 >= 1인 신호 개수
#
# 1은 탐지 임계값 확정이 아니라
# 비교용 진단선일 뿐임
# =====================================================

behavior_signal["a1_ge1_count"] = (
    behavior_signal[a1_cols] >= 1
).sum(axis=1)

behavior_signal["a2_ge1_count"] = (
    behavior_signal[a2_cols] >= 1
).sum(axis=1)


strong_coherent = behavior_signal[
    (behavior_signal["a1_ge1_count"] >= 2)
    &
    (behavior_signal["a2_ge1_count"] >= 2)
].copy()


print(
    "\n=== 6. 참고: A1에서 2개 이상 ≥1 + "
    "A2에서 2축 이상 ≥1 ==="
)

print("관측치 수:", len(strong_coherent))

if len(strong_coherent) == 0:
    print("없음")

else:
    print(
        strong_coherent[
            [
                "행정동명",
                "기준연월",
                "a1_ge1_count",
                "a2_ge1_count"
            ] + signal_cols
        ]
        .sort_values(
            ["기준연월", "행정동명"]
        )
        .round(3)
        .to_string(index=False)
    )

=== 1. 통합 행동신호 테이블 검증 ===
shape: (110, 11)
행정동 수: 22
월 수: 5
동×월 중복: 0
결측: 0

=== 2. A1 + A2 행동신호 상관관계 ===
                             low_comm_signal  low_weekday_outing_signal  low_holiday_outing_signal  very_low_outing_signal  very_low_outing_comm_signal  outside_activity_axis  home_stay_axis  communication_axis
low_comm_signal                        1.000                      0.026                     -0.120                   0.099                        0.308                  0.169          -0.078              -0.149
low_weekday_outing_signal              0.026                      1.000                     -0.034                   0.340                       -0.051                  0.398           0.592               0.002
low_holiday_outing_signal             -0.120                     -0.034                      1.000                  -0.104                        0.081                 -0.272          -0.125              -0.019
very_low_outing_signal                 0.099      

In [12]:
import pandas as pd
import numpy as np


# =====================================================
# 1. 외출 도메인 변수 정의
# =====================================================

outing_a1_cols = [
    "low_weekday_outing_signal",
    "low_holiday_outing_signal",
    "very_low_outing_signal"
]

outing_a2_cols = [
    "outside_activity_axis",
    "home_stay_axis"
]

outing_cols = outing_a1_cols + outing_a2_cols


outing = behavior_signal[
    [
        "행정동코드",
        "행정동명",
        "기준연월"
    ] + outing_cols
].copy()


# =====================================================
# 2. 기본 검증
# =====================================================

print("=== 1. 외출 도메인 기본 검증 ===")

print("shape:", outing.shape)

print(
    "동×월 중복:",
    outing.duplicated(
        ["행정동코드", "기준연월"]
    ).sum()
)

print(
    "결측:",
    outing[outing_cols]
    .isna()
    .sum()
    .sum()
)


# =====================================================
# 3. A1 ↔ A2 외출 도메인 교차 상관
# =====================================================

outing_corr = (
    outing[outing_cols]
    .corr()
)

cross_corr = (
    outing_corr
    .loc[
        outing_a1_cols,
        outing_a2_cols
    ]
)

print("\n=== 2. 외출 도메인 A1 ↔ A2 상관 ===")

print(
    cross_corr
    .round(3)
    .to_string()
)


# =====================================================
# 4. 단순 방향 정합성
#
# A1: 3개 중 몇 개가 +
# A2: 2개 중 몇 개가 +
#
# 아직 탐지 규칙 아님
# =====================================================

outing["a1_outing_positive_count"] = (
    outing[outing_a1_cols] > 0
).sum(axis=1)

outing["a2_outing_positive_count"] = (
    outing[outing_a2_cols] > 0
).sum(axis=1)


print("\n=== 3. 외출 도메인 단순 방향 교차표 ===")

print(
    pd.crosstab(
        outing["a1_outing_positive_count"],
        outing["a2_outing_positive_count"]
    )
    .to_string()
)


# =====================================================
# 5. 강도 기준 진단
#
# Z >= 1은 확정 임계값이 아니라
# 상대적으로 뚜렷한 변화 탐색용 기준선
# =====================================================

outing["a1_outing_ge1_count"] = (
    outing[outing_a1_cols] >= 1
).sum(axis=1)

outing["a2_outing_ge1_count"] = (
    outing[outing_a2_cols] >= 1
).sum(axis=1)


print("\n=== 4. 외출 도메인 Z>=1 개수 교차표 ===")

print(
    pd.crosstab(
        outing["a1_outing_ge1_count"],
        outing["a2_outing_ge1_count"]
    )
    .to_string()
)


# =====================================================
# 6. 외출 영역 정합 후보
#
# 조건:
# A1 외출 관심집단 중 1개 이상 >= 1
# AND
# A2 실제 행동축 중 1개 이상 >= 1
#
# 넓은 탐색 후보
# =====================================================

outing_candidate = outing[
    (outing["a1_outing_ge1_count"] >= 1)
    &
    (outing["a2_outing_ge1_count"] >= 1)
].copy()


print("\n=== 5. 외출 영역 정합 후보 ===")
print("관측치 수:", len(outing_candidate))

print(
    outing_candidate[
        [
            "행정동명",
            "기준연월",
            "a1_outing_ge1_count",
            "a2_outing_ge1_count"
        ] + outing_cols
    ]
    .sort_values(
        ["기준연월", "행정동명"]
    )
    .round(3)
    .to_string(index=False)
)


# =====================================================
# 7. 더 강한 정합성
#
# A1 외출지표 2개 이상 >= 1
# AND
# A2 두 행동축 모두 >= 1
#
# 최종 판정 기준이 아니라
# 강한 동시성 사례 탐색
# =====================================================

strong_outing = outing[
    (outing["a1_outing_ge1_count"] >= 2)
    &
    (outing["a2_outing_ge1_count"] == 2)
].copy()


print("\n=== 6. 외출 영역 강한 정합 사례 ===")
print("관측치 수:", len(strong_outing))

if len(strong_outing) == 0:

    print("없음")

else:

    print(
        strong_outing[
            [
                "행정동명",
                "기준연월",
                "a1_outing_ge1_count",
                "a2_outing_ge1_count"
            ] + outing_cols
        ]
        .sort_values(
            ["기준연월", "행정동명"]
        )
        .round(3)
        .to_string(index=False)
    )


# =====================================================
# 8. 반복 출현 여부
#
# 특정 동이 여러 달에 걸쳐 후보로 등장하는지 확인
# =====================================================

repeat_candidate = (
    outing_candidate
    .groupby(
        ["행정동코드", "행정동명"]
    )
    .agg(
        candidate_month_count=(
            "기준연월",
            "count"
        ),
        candidate_months=(
            "기준연월",
            lambda x: ", ".join(
                sorted(x.astype(str))
            )
        )
    )
    .sort_values(
        "candidate_month_count",
        ascending=False
    )
    .reset_index()
)


print("\n=== 7. 외출 정합 후보 반복 출현 ===")

print(
    repeat_candidate
    .to_string(index=False)
)

=== 1. 외출 도메인 기본 검증 ===
shape: (110, 8)
동×월 중복: 0
결측: 0

=== 2. 외출 도메인 A1 ↔ A2 상관 ===
                           outside_activity_axis  home_stay_axis
low_weekday_outing_signal                  0.398           0.592
low_holiday_outing_signal                 -0.272          -0.125
very_low_outing_signal                     0.476           0.541

=== 3. 외출 도메인 단순 방향 교차표 ===
a2_outing_positive_count   0   1   2
a1_outing_positive_count            
0                         13   3   0
1                         19  11   8
2                          8  16  17
3                          2   3  10

=== 4. 외출 도메인 Z>=1 개수 교차표 ===
a2_outing_ge1_count   0  1  2
a1_outing_ge1_count          
0                    58  9  2
1                    18  9  5
2                     5  0  3
3                     1  0  0

=== 5. 외출 영역 정합 후보 ===
관측치 수: 17
행정동명    기준연월  a1_outing_ge1_count  a2_outing_ge1_count  low_weekday_outing_signal  low_holiday_outing_signal  very_low_outing_signal  outside_activity_axis  h

In [13]:
import pandas as pd
import numpy as np


# =====================================================
# 1. 핵심 외출신호 정의
# =====================================================

core_outing_cols = [
    "low_weekday_outing_signal",
    "very_low_outing_signal",
    "outside_activity_axis",
    "home_stay_axis"
]

aux_outing_col = "low_holiday_outing_signal"


outing_core = behavior_signal[
    [
        "행정동코드",
        "행정동명",
        "기준연월"
    ]
    + core_outing_cols
    + [aux_outing_col]
].copy()


# =====================================================
# 2. 각 핵심신호가 탐색선 >= 1인지 표시
# =====================================================

flag_cols = []

for col in core_outing_cols:

    flag_col = f"{col}__ge1"

    outing_core[flag_col] = (
        outing_core[col] >= 1
    ).astype(int)

    flag_cols.append(flag_col)


# 한 동×월에서 핵심 4개 중 몇 개가 >=1인가
outing_core["core_signal_count"] = (
    outing_core[flag_cols].sum(axis=1)
)


print("=== 1. 핵심 외출신호 기본 검증 ===")

print("shape:", outing_core.shape)

print(
    "동×월 중복:",
    outing_core.duplicated(
        ["행정동코드", "기준연월"]
    ).sum()
)

print(
    "결측:",
    outing_core[
        core_outing_cols
        + [aux_outing_col]
    ].isna().sum().sum()
)


# =====================================================
# 3. 동×월별 핵심신호 개수 분포
# =====================================================

print("\n=== 2. 핵심 4신호 중 >=1 개수 분포 ===")

print(
    outing_core["core_signal_count"]
    .value_counts()
    .sort_index()
    .to_string()
)


# =====================================================
# 4. 각 행정동에서 각 신호가 몇 개월 등장했는지
# =====================================================

signal_frequency = (
    outing_core
    .groupby(
        ["행정동코드", "행정동명"]
    )[flag_cols]
    .sum()
    .reset_index()
)

signal_frequency = signal_frequency.rename(
    columns={
        "low_weekday_outing_signal__ge1":
            "평일저외출_등장월수",

        "very_low_outing_signal__ge1":
            "극저외출_등장월수",

        "outside_activity_axis__ge1":
            "외부활동감소_등장월수",

        "home_stay_axis__ge1":
            "집체류증가_등장월수"
    }
)


print("\n=== 3. 행정동별 핵심신호 등장 월수 ===")

print(
    signal_frequency
    .sort_values(
        [
            "평일저외출_등장월수",
            "극저외출_등장월수",
            "외부활동감소_등장월수",
            "집체류증가_등장월수"
        ],
        ascending=False
    )
    .to_string(index=False)
)


# =====================================================
# 5. '정합 월' 정의
#
# 관심집단 계열(A1):
# 평일저외출 또는 극저외출 중 하나 이상 >=1
#
# 직접행동 계열(A2):
# 외부활동감소 또는 집체류증가 중 하나 이상 >=1
#
# 두 계열이 같은 달 함께 나타나는지만 확인
# =====================================================

outing_core["a1_core_present"] = (
    (
        outing_core[
            "low_weekday_outing_signal"
        ] >= 1
    )
    |
    (
        outing_core[
            "very_low_outing_signal"
        ] >= 1
    )
).astype(int)


outing_core["a2_core_present"] = (
    (
        outing_core[
            "outside_activity_axis"
        ] >= 1
    )
    |
    (
        outing_core[
            "home_stay_axis"
        ] >= 1
    )
).astype(int)


outing_core["core_coherent_month"] = (
    (
        outing_core["a1_core_present"] == 1
    )
    &
    (
        outing_core["a2_core_present"] == 1
    )
).astype(int)


# =====================================================
# 6. 행정동별 정합 월수
# =====================================================

coherent_frequency = (
    outing_core
    .groupby(
        ["행정동코드", "행정동명"]
    )
    .agg(
        coherent_month_count=(
            "core_coherent_month",
            "sum"
        ),
        coherent_months=(
            "기준연월",
            lambda x: ", ".join(
                outing_core.loc[
                    x.index[
                        outing_core.loc[
                            x.index,
                            "core_coherent_month"
                        ].values == 1
                    ],
                    "기준연월"
                ].astype(str)
            )
        )
    )
    .reset_index()
)


print("\n=== 4. 행정동별 핵심 외출 정합 월수 ===")

print(
    coherent_frequency
    .sort_values(
        "coherent_month_count",
        ascending=False
    )
    .to_string(index=False)
)


# =====================================================
# 7. 정합 월의 세부 구성
# =====================================================

coherent_detail = outing_core[
    outing_core["core_coherent_month"] == 1
].copy()


print("\n=== 5. 핵심 외출 정합 월 세부 구성 ===")

print(
    coherent_detail[
        [
            "행정동명",
            "기준연월",
            "low_weekday_outing_signal",
            "very_low_outing_signal",
            "outside_activity_axis",
            "home_stay_axis",
            "low_holiday_outing_signal",
            "core_signal_count"
        ]
    ]
    .sort_values(
        ["행정동명", "기준연월"]
    )
    .round(3)
    .to_string(index=False)
)


# =====================================================
# 8. 2개월 이상 반복된 행정동의 월별 전체 궤적
#
# 후보 월뿐 아니라 8~12월 전체를 출력해야
# 중간에 신호가 끊겼는지도 볼 수 있음
# =====================================================

repeat_dongs = (
    coherent_frequency.loc[
        coherent_frequency[
            "coherent_month_count"
        ] >= 2,
        "행정동명"
    ]
    .tolist()
)


trajectory = outing_core[
    outing_core["행정동명"].isin(
        repeat_dongs
    )
].copy()


print("\n=== 6. 정합 월 2회 이상 행정동의 8~12월 전체 궤적 ===")

if len(repeat_dongs) == 0:

    print("해당 행정동 없음")

else:

    for dong in repeat_dongs:

        print(f"\n[{dong}]")

        temp = trajectory[
            trajectory["행정동명"] == dong
        ]

        print(
            temp[
                [
                    "기준연월",
                    "low_weekday_outing_signal",
                    "very_low_outing_signal",
                    "outside_activity_axis",
                    "home_stay_axis",
                    "low_holiday_outing_signal",
                    "core_signal_count",
                    "core_coherent_month"
                ]
            ]
            .sort_values("기준연월")
            .round(3)
            .to_string(index=False)
        )

=== 1. 핵심 외출신호 기본 검증 ===
shape: (110, 13)
동×월 중복: 0
결측: 0

=== 2. 핵심 4신호 중 >=1 개수 분포 ===
core_signal_count
0    68
1    21
2    13
3     6
4     2

=== 3. 행정동별 핵심신호 등장 월수 ===
  행정동코드 행정동명  평일저외출_등장월수  극저외출_등장월수  외부활동감소_등장월수  집체류증가_등장월수
1123051  신사동           3          2            1           2
1123073 일원1동           3          2            1           2
1123072 일원본동           3          0            3           3
1123063 대치4동           2          1            2           1
1123064 역삼1동           1          1            1           1
1123067 도곡2동           1          1            1           0
1123079 대치2동           1          1            1           0
1123059 삼성2동           1          1            0           0
1123077 압구정동           1          1            0           0
1123058 삼성1동           1          0            1           1
1123080 개포2동           1          0            1           1
1123076  세곡동           1          0            0           0
1123074 개포3동           0        

In [14]:
import pandas as pd
import numpy as np


# =====================================================
# 1. 소통 도메인 데이터 구성
# =====================================================

comm_cols = [
    "low_comm_signal",
    "very_low_outing_comm_signal",
    "communication_axis"
]

comm = behavior_signal[
    [
        "행정동코드",
        "행정동명",
        "기준연월"
    ] + comm_cols
].copy()


print("=== 1. 소통 도메인 기본 검증 ===")

print("shape:", comm.shape)

print(
    "동×월 중복:",
    comm.duplicated(
        ["행정동코드", "기준연월"]
    ).sum()
)

print(
    "결측:",
    comm[comm_cols]
    .isna()
    .sum()
    .sum()
)


# =====================================================
# 2. 세 신호의 상관관계
# =====================================================

print("\n=== 2. 소통 도메인 상관관계 ===")

print(
    comm[comm_cols]
    .corr()
    .round(3)
    .to_string()
)


# =====================================================
# 3. 단순 방향 정합성
#
# A1 관심집단:
# 두 변수 중 양(+)인 개수
#
# A2:
# communication_axis > 0 여부
# =====================================================

comm["a1_comm_positive_count"] = (
    comm[
        [
            "low_comm_signal",
            "very_low_outing_comm_signal"
        ]
    ] > 0
).sum(axis=1)

comm["a2_comm_positive"] = (
    comm["communication_axis"] > 0
).astype(int)


print("\n=== 3. 소통 단순 방향 정합성 ===")

print(
    pd.crosstab(
        comm["a1_comm_positive_count"],
        comm["a2_comm_positive"]
    )
    .to_string()
)


# =====================================================
# 4. Z >= 1 탐색선
#
# 최종 임계값이 아니라
# 비교를 위한 진단선
# =====================================================

comm["low_comm_ge1"] = (
    comm["low_comm_signal"] >= 1
).astype(int)

comm["very_low_outing_comm_ge1"] = (
    comm["very_low_outing_comm_signal"] >= 1
).astype(int)

comm["communication_ge1"] = (
    comm["communication_axis"] >= 1
).astype(int)

comm["a1_comm_ge1_count"] = (
    comm[
        [
            "low_comm_ge1",
            "very_low_outing_comm_ge1"
        ]
    ].sum(axis=1)
)


print("\n=== 4. 소통 Z>=1 교차표 ===")

print(
    pd.crosstab(
        comm["a1_comm_ge1_count"],
        comm["communication_ge1"]
    )
    .to_string()
)


# =====================================================
# 5. A1 ↔ A2 정합 후보
#
# A1 두 관심집단 중 하나 이상 >=1
# AND
# communication_axis >=1
# =====================================================

comm_candidate = comm[
    (comm["a1_comm_ge1_count"] >= 1)
    &
    (comm["communication_ge1"] == 1)
].copy()


print("\n=== 5. 소통 정합 후보 ===")
print("관측치 수:", len(comm_candidate))

if len(comm_candidate) == 0:

    print("없음")

else:

    print(
        comm_candidate[
            [
                "행정동명",
                "기준연월",
                "low_comm_signal",
                "very_low_outing_comm_signal",
                "communication_axis",
                "a1_comm_ge1_count"
            ]
        ]
        .sort_values(
            ["기준연월", "행정동명"]
        )
        .round(3)
        .to_string(index=False)
    )


# =====================================================
# 6. 더 강한 정합 사례
#
# 두 A1 지표 모두 >=1
# AND
# communication_axis >=1
#
# 역시 최종 탐지규칙 아님
# =====================================================

strong_comm = comm[
    (comm["a1_comm_ge1_count"] == 2)
    &
    (comm["communication_ge1"] == 1)
].copy()


print("\n=== 6. 소통 강한 정합 사례 ===")
print("관측치 수:", len(strong_comm))

if len(strong_comm) == 0:

    print("없음")

else:

    print(
        strong_comm[
            [
                "행정동명",
                "기준연월",
                "low_comm_signal",
                "very_low_outing_comm_signal",
                "communication_axis"
            ]
        ]
        .sort_values(
            ["기준연월", "행정동명"]
        )
        .round(3)
        .to_string(index=False)
    )


# =====================================================
# 7. 행정동별 반복 출현
# =====================================================

if len(comm_candidate) > 0:

    comm_repeat = (
        comm_candidate
        .groupby(
            ["행정동코드", "행정동명"]
        )
        .agg(
            coherent_month_count=(
                "기준연월",
                "count"
            ),
            coherent_months=(
                "기준연월",
                lambda x: ", ".join(
                    sorted(x.astype(str))
                )
            )
        )
        .reset_index()
        .sort_values(
            "coherent_month_count",
            ascending=False
        )
    )

    print("\n=== 7. 소통 정합 후보 반복 출현 ===")

    print(
        comm_repeat
        .to_string(index=False)
    )


# =====================================================
# 8. 반대 방향 사례 진단
#
# A1 저소통 관심집단은 강하게 증가했지만
# 실제 통화대상자 감소축은 반대인 경우
#
# A1 >=1
# communication_axis <= -1
# =====================================================

discordant_comm = comm[
    (comm["a1_comm_ge1_count"] >= 1)
    &
    (comm["communication_axis"] <= -1)
].copy()


print("\n=== 8. 소통 반대 방향 사례 ===")
print("관측치 수:", len(discordant_comm))

if len(discordant_comm) == 0:

    print("없음")

else:

    print(
        discordant_comm[
            [
                "행정동명",
                "기준연월",
                "low_comm_signal",
                "very_low_outing_comm_signal",
                "communication_axis",
                "a1_comm_ge1_count"
            ]
        ]
        .sort_values(
            ["기준연월", "행정동명"]
        )
        .round(3)
        .to_string(index=False)
    )

=== 1. 소통 도메인 기본 검증 ===
shape: (110, 6)
동×월 중복: 0
결측: 0

=== 2. 소통 도메인 상관관계 ===
                             low_comm_signal  very_low_outing_comm_signal  communication_axis
low_comm_signal                        1.000                        0.308              -0.149
very_low_outing_comm_signal            0.308                        1.000               0.040
communication_axis                    -0.149                        0.040               1.000

=== 3. 소통 단순 방향 정합성 ===
a2_comm_positive         0   1
a1_comm_positive_count        
0                       16  16
1                       18  28
2                       21  11

=== 4. 소통 Z>=1 교차표 ===
communication_ge1   0  1
a1_comm_ge1_count       
0                  72  9
1                  21  5
2                   3  0

=== 5. 소통 정합 후보 ===
관측치 수: 5
행정동명    기준연월  low_comm_signal  very_low_outing_comm_signal  communication_axis  a1_comm_ge1_count
대치4동 2025-09           -0.613                        1.012               2.859         

In [16]:
from google.colab import files
import pandas as pd
import io

# 파일 업로드
uploaded = files.upload()

# 업로드한 파일명 확인
print(uploaded.keys())

Saving gangnam_db1_dong_feature_mart_2025.csv to gangnam_db1_dong_feature_mart_2025.csv
dict_keys(['gangnam_db1_dong_feature_mart_2025.csv'])


In [17]:
filename = "gangnam_db1_dong_feature_mart_2025.csv"

dong_feature_mart = pd.read_csv(
    io.BytesIO(uploaded[filename]),
    encoding="utf-8-sig"
)

print("shape:", dong_feature_mart.shape)
print("행정동 수:", dong_feature_mart["행정동코드"].nunique())
print("중복:", dong_feature_mart["행정동코드"].duplicated().sum())

print("\n컬럼:")
print(dong_feature_mart.columns.tolist())

shape: (22, 75)
행정동 수: 22
중복: 0

컬럼:
['행정동코드', '행정동명', 'total_flow_mean', 'flow_per_point_mean', 'male_ratio_mean', 'age_20_ratio_mean', 'age_30_ratio_mean', 'age_40_ratio_mean', 'age_50_ratio_mean', 'age_60_plus_ratio_mean', 'morning_ratio_mean', 'daytime_ratio_mean', 'evening_ratio_mean', 'peak_hour_ratio_mean', 'weekend_weekday_index_mean', 'peak_day_ratio_mean', 'low_comm_ratio_mean', 'low_weekday_outing_ratio_mean', 'low_holiday_outing_ratio_mean', 'high_worktime_ratio_mean', 'high_video_ratio_mean', 'high_life_service_ratio_mean', 'financial_interest_ratio_mean', 'very_low_outing_ratio_mean', 'very_low_outing_comm_ratio_mean', '야간상주지 변경횟수 평균_mean', '평균 근무시간 평균_mean', '평일 총 이동 횟수_mean', '휴일 총 이동 횟수 평균_mean', '집 추정 위치 평일 총 체류시간_mean', '집 추정 위치 휴일 총 체류시간_mean', '금융 서비스 사용일수_mean', '쇼핑 서비스 사용일수_mean', '배달 서비스 사용일수_mean', '최근 3개월 내 요금 연체 비율_mean', '평균 통화대상자 수_mean', '데이터 사용량_mean', '카카오톡_비사용비율_mean', 'resident_total_mean', 'foreigner_ratio_mean', 'resident_age_10_ratio', 'resident_age

In [18]:
# =====================================================
# 1. Context 변수 명시적 정의
# =====================================================

resident_context = [
    "resident_total_mean",
    "foreigner_ratio_mean",
    "resident_age_10_ratio",
    "resident_age_20_ratio",
    "resident_age_30_ratio",
    "resident_age_40_ratio",
    "resident_age_50_ratio",
    "resident_age_60_plus_ratio"
]

single_context = [
    "single_age_20_gap",
    "single_age_30_gap",
    "single_age_40_gap",
    "single_age_50_gap",
    "single_age_60_plus_gap",
    "single_gender_1_ratio",
    "single_gender_2_ratio",
    "single_household_mean",
    "official_single_household_mean",
    "official_single_household_change_rate"
]

welfare_context = [
    "livelihood_per_1000",
    "medical_per_1000",
    "housing_per_1000",
    "elderly_livelihood_per_1000",
    "elderly_medical_per_1000",
    "elderly_housing_per_1000"
]

disability_context = [
    "disabled_total",
    "disabled_per_1000",
    "disabled_male_ratio",
    "disabled_female_ratio",
    "brain_lesion_ratio",
    "visual_ratio",
    "kidney_ratio",
    "autism_ratio",
    "mental_ratio",
    "intellectual_ratio",
    "physical_ratio",
    "hearing_ratio",
    "other_disability_ratio"
]

context_cols = (
    resident_context
    + single_context
    + welfare_context
    + disability_context
)


# =====================================================
# 2. 변수 존재 여부 확인
# =====================================================

missing_cols = [
    col for col in context_cols
    if col not in dong_feature_mart.columns
]

print("=== 1. Context 변수 확인 ===")
print("Context 변수 수:", len(context_cols))
print("없는 변수:", missing_cols)


# =====================================================
# 3. 행동신호 요약이 현재 런타임에 있는지 확인
# =====================================================

print(
    "\ncontext_signal_summary 존재:",
    "context_signal_summary" in globals()
)


# =====================================================
# 4. Context Mart 생성
# =====================================================

if "context_signal_summary" in globals():

    context_mart = (
        dong_feature_mart[
            [
                "행정동코드",
                "행정동명"
            ] + context_cols
        ]
        .merge(
            context_signal_summary,
            on=[
                "행정동코드",
                "행정동명"
            ],
            how="left",
            validate="one_to_one"
        )
    )

    print("\n=== 2. Context Mart 검증 ===")
    print("shape:", context_mart.shape)
    print(
        "행정동 수:",
        context_mart["행정동코드"].nunique()
    )
    print(
        "중복:",
        context_mart.duplicated(
            ["행정동코드"]
        ).sum()
    )

    print("\nContext 영역별 변수 수")
    print("주민인구:", len(resident_context))
    print("1인가구:", len(single_context))
    print("복지·경제:", len(welfare_context))
    print("장애:", len(disability_context))

    print("\nContext 영역별 결측")
    print(
        "주민인구:",
        context_mart[resident_context]
        .isna().sum().sum()
    )
    print(
        "1인가구:",
        context_mart[single_context]
        .isna().sum().sum()
    )
    print(
        "복지·경제:",
        context_mart[welfare_context]
        .isna().sum().sum()
    )
    print(
        "장애:",
        context_mart[disability_context]
        .isna().sum().sum()
    )


# =====================================================
# 5. 반복 행동패턴 지역 확인
# =====================================================

if "context_mart" in globals():

    print("\n=== 3. 반복 외출 행동패턴 지역 ===")

    print(
        context_mart.loc[
            context_mart[
                "coherent_month_count"
            ] >= 2,
            [
                "행정동명",
                "coherent_month_count",
                "avg_low_weekday_outing_signal",
                "avg_very_low_outing_signal",
                "avg_outside_activity_axis",
                "avg_home_stay_axis"
            ]
        ]
        .sort_values(
            "coherent_month_count",
            ascending=False
        )
        .round(3)
        .to_string(index=False)
    )

=== 1. Context 변수 확인 ===
Context 변수 수: 37
없는 변수: []

context_signal_summary 존재: True

=== 2. Context Mart 검증 ===
shape: (22, 46)
행정동 수: 22
중복: 0

Context 영역별 변수 수
주민인구: 8
1인가구: 10
복지·경제: 6
장애: 13

Context 영역별 결측
주민인구: 0
1인가구: 0
복지·경제: 2
장애: 0

=== 3. 반복 외출 행동패턴 지역 ===
행정동명  coherent_month_count  avg_low_weekday_outing_signal  avg_very_low_outing_signal  avg_outside_activity_axis  avg_home_stay_axis
일원본동                     3                          1.138                      -0.078                      0.238               0.541
개포3동                     3                         -0.236                       1.792                      0.839               1.377
 신사동                     2                          0.771                       0.472                     -0.109               0.393
일원1동                     2                          1.157                       1.054                     -0.020               1.049


In [19]:
import pandas as pd
import numpy as np


# =====================================================
# 1. 반복 행동패턴 지역
# =====================================================

repeat_dongs = (
    context_mart.loc[
        context_mart["coherent_month_count"] >= 2,
        "행정동명"
    ]
    .tolist()
)

print("=== 1. 반복 행동패턴 지역 ===")
print(repeat_dongs)


# =====================================================
# 2. 주민인구 Context percentile 계산
#
# percentile이 높음 =
# 강남구 22개 동 중 해당 변수 값이 상대적으로 높음
# 위험도 의미 아님
# =====================================================

resident_percentile = context_mart[
    [
        "행정동코드",
        "행정동명"
    ] + resident_context
].copy()


for col in resident_context:

    resident_percentile[
        f"{col}__pct"
    ] = (
        resident_percentile[col]
        .rank(
            pct=True,
            method="average"
        )
        * 100
    )


# =====================================================
# 3. 반복지역 실제값 + percentile
# =====================================================

print(
    "\n=== 2. 반복지역 주민인구 Context "
    "실제값 / Percentile ==="
)

for dong in repeat_dongs:

    print(f"\n[{dong}]")

    row = resident_percentile[
        resident_percentile["행정동명"] == dong
    ].iloc[0]

    result = []

    for col in resident_context:

        result.append({
            "feature": col,
            "value": row[col],
            "percentile": row[
                f"{col}__pct"
            ]
        })

    print(
        pd.DataFrame(result)
        .round(4)
        .to_string(index=False)
    )


# =====================================================
# 4. 네 반복지역의 percentile 비교표
# =====================================================

comparison_rows = []

for dong in repeat_dongs:

    row = resident_percentile[
        resident_percentile["행정동명"] == dong
    ].iloc[0]

    temp = {
        "행정동명": dong
    }

    for col in resident_context:

        temp[col] = row[
            f"{col}__pct"
        ]

    comparison_rows.append(temp)


resident_repeat_pct = pd.DataFrame(
    comparison_rows
)


print(
    "\n=== 3. 반복지역 주민인구 Percentile 비교 ==="
)

print(
    resident_repeat_pct
    .round(1)
    .to_string(index=False)
)


# =====================================================
# 5. 참고용:
# 반복 4개 동 vs 나머지 18개 동의 중앙값
#
# 표본이 작으므로 검정/인과해석 목적 아님
# =====================================================

temp = context_mart.copy()

temp["repeat_group"] = np.where(
    temp["coherent_month_count"] >= 2,
    "반복 4개 동",
    "기타 18개 동"
)


group_median = (
    temp
    .groupby("repeat_group")[
        resident_context
    ]
    .median()
    .T
)

group_median["difference"] = (
    group_median["반복 4개 동"]
    - group_median["기타 18개 동"]
)


print(
    "\n=== 4. 참고: 반복 4개 동 vs 기타 18개 동 중앙값 ==="
)

print(
    group_median
    .round(4)
    .to_string()
)

=== 1. 반복 행동패턴 지역 ===
['신사동', '일원본동', '일원1동', '개포3동']

=== 2. 반복지역 주민인구 Context 실제값 / Percentile ===

[신사동]
                   feature      value  percentile
       resident_total_mean 15309.0000     18.1818
      foreigner_ratio_mean     0.0138     77.2727
     resident_age_10_ratio     0.1006     40.9091
     resident_age_20_ratio     0.1250     68.1818
     resident_age_30_ratio     0.1377     36.3636
     resident_age_40_ratio     0.1664     18.1818
     resident_age_50_ratio     0.1822     72.7273
resident_age_60_plus_ratio     0.2881     77.2727

[일원본동]
                   feature      value  percentile
       resident_total_mean 22431.5000     45.4545
      foreigner_ratio_mean     0.0010      4.5455
     resident_age_10_ratio     0.1564     86.3636
     resident_age_20_ratio     0.0970     18.1818
     resident_age_30_ratio     0.1221     22.7273
     resident_age_40_ratio     0.2043     77.2727
     resident_age_50_ratio     0.1671     36.3636
resident_age_60_plus_ratio     0.2

In [20]:
# =====================================================
# 1. 1인가구 Context percentile
# =====================================================

single_percentile = context_mart[
    [
        "행정동코드",
        "행정동명"
    ] + single_context
].copy()


for col in single_context:

    single_percentile[
        f"{col}__pct"
    ] = (
        single_percentile[col]
        .rank(
            pct=True,
            method="average"
        )
        * 100
    )


# =====================================================
# 2. 반복지역 실제값 + percentile
# =====================================================

print(
    "=== 1. 반복지역 1인가구 Context "
    "실제값 / Percentile ==="
)

for dong in repeat_dongs:

    print(f"\n[{dong}]")

    row = single_percentile[
        single_percentile["행정동명"] == dong
    ].iloc[0]

    result = []

    for col in single_context:

        result.append({
            "feature": col,
            "value": row[col],
            "percentile": row[
                f"{col}__pct"
            ]
        })

    print(
        pd.DataFrame(result)
        .round(4)
        .to_string(index=False)
    )


# =====================================================
# 3. 네 반복지역 percentile 비교
# =====================================================

comparison_rows = []

for dong in repeat_dongs:

    row = single_percentile[
        single_percentile["행정동명"] == dong
    ].iloc[0]

    temp = {"행정동명": dong}

    for col in single_context:

        temp[col] = row[
            f"{col}__pct"
        ]

    comparison_rows.append(temp)


single_repeat_pct = pd.DataFrame(
    comparison_rows
)


print(
    "\n=== 2. 반복지역 1인가구 Percentile 비교 ==="
)

print(
    single_repeat_pct
    .round(1)
    .to_string(index=False)
)


# =====================================================
# 4. 반복 4개 동 vs 기타 18개 동 중앙값
# 참고용 기술통계
# =====================================================

temp = context_mart.copy()

temp["repeat_group"] = np.where(
    temp["coherent_month_count"] >= 2,
    "반복 4개 동",
    "기타 18개 동"
)


single_group_median = (
    temp
    .groupby("repeat_group")[
        single_context
    ]
    .median()
    .T
)

single_group_median["difference"] = (
    single_group_median["반복 4개 동"]
    - single_group_median["기타 18개 동"]
)


print(
    "\n=== 3. 참고: 반복 4개 동 vs "
    "기타 18개 동 중앙값 ==="
)

print(
    single_group_median
    .round(4)
    .to_string()
)

=== 1. 반복지역 1인가구 Context 실제값 / Percentile ===

[신사동]
                              feature     value  percentile
                    single_age_20_gap    0.0570     90.9091
                    single_age_30_gap    0.1212     81.8182
                    single_age_40_gap   -0.0528     45.4545
                    single_age_50_gap   -0.1076     13.6364
               single_age_60_plus_gap   -0.0177     31.8182
                single_gender_1_ratio    0.4562     27.2727
                single_gender_2_ratio    0.5438     77.2727
                single_household_mean 2430.5467     36.3636
       official_single_household_mean 2447.5000     27.2727
official_single_household_change_rate   -0.0029     72.7273

[일원본동]
                              feature     value  percentile
                    single_age_20_gap   -0.0298     27.2727
                    single_age_30_gap   -0.0451      9.0909
                    single_age_40_gap   -0.1323      9.0909
                    single_age_50_gap  

In [21]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import pandas as pd
import numpy as np


# =====================================================
# 1. 복지·경제 원자료 Percentile
# =====================================================

welfare_percentile = context_mart[
    ["행정동코드", "행정동명"] + welfare_context
].copy()

for col in welfare_context:
    welfare_percentile[f"{col}__pct"] = (
        welfare_percentile[col]
        .rank(
            pct=True,
            method="average"
        )
        * 100
    )


print("=== 1. 반복지역 복지·경제 Context 실제값 / Percentile ===")

for dong in repeat_dongs:

    print(f"\n[{dong}]")

    row = welfare_percentile[
        welfare_percentile["행정동명"] == dong
    ].iloc[0]

    result = []

    for col in welfare_context:
        result.append({
            "feature": col,
            "value": row[col],
            "percentile": row[f"{col}__pct"]
        })

    print(
        pd.DataFrame(result)
        .round(4)
        .to_string(index=False)
    )


# =====================================================
# 2. 6개 지표 PCA
#
# 결측은 PCA 계산에 한해서만
# 각 변수의 22개 동 중앙값으로 임시 대체
# =====================================================

welfare_for_pca = context_mart[
    welfare_context
].copy()

for col in welfare_context:
    welfare_for_pca[col] = (
        welfare_for_pca[col]
        .fillna(
            welfare_for_pca[col].median()
        )
    )

scaler = StandardScaler()

X_welfare = scaler.fit_transform(
    welfare_for_pca
)

pca = PCA()

welfare_pcs = pca.fit_transform(
    X_welfare
)


print("\n=== 2. Welfare PCA 설명분산 ===")

for i, ratio in enumerate(
    pca.explained_variance_ratio_,
    start=1
):
    print(
        f"PC{i}: "
        f"{ratio:.4f} "
        f"({ratio * 100:.2f}%)"
    )


# =====================================================
# 3. PC1 Loading 확인
# =====================================================

pc1_loadings = pd.DataFrame({
    "feature": welfare_context,
    "PC1_loading": pca.components_[0]
})


# PCA 부호는 임의적이므로
# 평균 loading이 음수라면 방향 통일
if pc1_loadings["PC1_loading"].mean() < 0:

    welfare_pcs[:, 0] *= -1
    pca.components_[0] *= -1

    pc1_loadings["PC1_loading"] = (
        pca.components_[0]
    )


print("\n=== 3. Welfare PC1 Loading ===")

print(
    pc1_loadings
    .round(4)
    .to_string(index=False)
)


# =====================================================
# 4. Context Mart에 PC1 추가
# =====================================================

context_mart[
    "welfare_context_pc1"
] = welfare_pcs[:, 0]


context_mart[
    "welfare_context_pc1_percentile"
] = (
    context_mart[
        "welfare_context_pc1"
    ]
    .rank(
        pct=True,
        method="average"
    )
    * 100
)


# =====================================================
# 5. 반복지역 PC1 비교
# =====================================================

print(
    "\n=== 4. 반복지역 Welfare PC1 ==="
)

print(
    context_mart.loc[
        context_mart["행정동명"].isin(
            repeat_dongs
        ),
        [
            "행정동명",
            "coherent_month_count",
            "welfare_context_pc1",
            "welfare_context_pc1_percentile"
        ]
    ]
    .sort_values(
        "welfare_context_pc1",
        ascending=False
    )
    .round(3)
    .to_string(index=False)
)


# =====================================================
# 6. 반복 4개 동 vs 기타 18개 동
# PC1 중앙값 비교
# =====================================================

context_mart[
    "repeat_group"
] = np.where(
    context_mart[
        "coherent_month_count"
    ] >= 2,
    "반복 4개 동",
    "기타 18개 동"
)


print(
    "\n=== 5. Welfare PC1 그룹 중앙값 ==="
)

print(
    context_mart
    .groupby("repeat_group")[
        "welfare_context_pc1"
    ]
    .median()
    .round(4)
    .to_string()
)


# =====================================================
# 7. 네 반복지역의 원자료 Percentile 비교표
# =====================================================

comparison_rows = []

for dong in repeat_dongs:

    row = welfare_percentile[
        welfare_percentile["행정동명"] == dong
    ].iloc[0]

    temp = {"행정동명": dong}

    for col in welfare_context:
        temp[col] = row[
            f"{col}__pct"
        ]

    comparison_rows.append(temp)


welfare_repeat_pct = pd.DataFrame(
    comparison_rows
)


print(
    "\n=== 6. 반복지역 복지·경제 Percentile 비교 ==="
)

print(
    welfare_repeat_pct
    .round(1)
    .to_string(index=False)
)

=== 1. 반복지역 복지·경제 Context 실제값 / Percentile ===

[신사동]
                    feature   value  percentile
        livelihood_per_1000  7.7732     50.0000
           medical_per_1000  0.8492     57.1429
           housing_per_1000  5.6829     59.0909
elderly_livelihood_per_1000 16.3808     50.0000
   elderly_medical_per_1000  2.7816     66.6667
   elderly_housing_per_1000  8.3449     50.0000

[일원본동]
                    feature  value  percentile
        livelihood_per_1000 3.6556     27.2727
           medical_per_1000 0.1337     14.2857
           housing_per_1000 2.8977     36.3636
elderly_livelihood_per_1000 8.1946     22.7273
   elderly_medical_per_1000 0.7682     33.3333
   elderly_housing_per_1000 4.8656     27.2727

[일원1동]
                    feature    value  percentile
        livelihood_per_1000  83.8038     95.4545
           medical_per_1000   6.9427     95.2381
           housing_per_1000  30.5761     95.4545
elderly_livelihood_per_1000 168.9359     90.9091
   elderly_medical_p

In [22]:
# =====================================================
# 1. 장애 Context Percentile 계산
# =====================================================

disability_percentile = context_mart[
    ["행정동코드", "행정동명"] + disability_context
].copy()

for col in disability_context:

    disability_percentile[
        f"{col}__pct"
    ] = (
        disability_percentile[col]
        .rank(
            pct=True,
            method="average"
        )
        * 100
    )


# =====================================================
# 2. 반복지역 실제값 + percentile
# =====================================================

print(
    "=== 1. 반복지역 장애 Context "
    "실제값 / Percentile ==="
)

for dong in repeat_dongs:

    print(f"\n[{dong}]")

    row = disability_percentile[
        disability_percentile["행정동명"] == dong
    ].iloc[0]

    result = []

    for col in disability_context:

        result.append({
            "feature": col,
            "value": row[col],
            "percentile": row[
                f"{col}__pct"
            ]
        })

    print(
        pd.DataFrame(result)
        .round(4)
        .to_string(index=False)
    )


# =====================================================
# 3. 반복지역 Percentile 비교표
# =====================================================

comparison_rows = []

for dong in repeat_dongs:

    row = disability_percentile[
        disability_percentile["행정동명"] == dong
    ].iloc[0]

    temp = {"행정동명": dong}

    for col in disability_context:

        temp[col] = row[
            f"{col}__pct"
        ]

    comparison_rows.append(temp)


disability_repeat_pct = pd.DataFrame(
    comparison_rows
)


print(
    "\n=== 2. 반복지역 장애 Context "
    "Percentile 비교 ==="
)

print(
    disability_repeat_pct
    .round(1)
    .to_string(index=False)
)


# =====================================================
# 4. 반복 4개 동 vs 기타 18개 동 중앙값
# =====================================================

temp = context_mart.copy()

temp["repeat_group"] = np.where(
    temp["coherent_month_count"] >= 2,
    "반복 4개 동",
    "기타 18개 동"
)


disability_group_median = (
    temp
    .groupby("repeat_group")[
        disability_context
    ]
    .median()
    .T
)

disability_group_median["difference"] = (
    disability_group_median["반복 4개 동"]
    - disability_group_median["기타 18개 동"]
)


print(
    "\n=== 3. 반복 4개 동 vs "
    "기타 18개 동 중앙값 ==="
)

print(
    disability_group_median
    .round(4)
    .to_string()
)


# =====================================================
# 5. 핵심 규모지표만 별도 확인
#
# disabled_total은 지역 인구규모 영향을 받으므로
# disabled_per_1000을 상대비교의 중심으로 봄
# =====================================================

print(
    "\n=== 4. 반복지역 장애 규모 핵심지표 ==="
)

print(
    context_mart.loc[
        context_mart["행정동명"].isin(
            repeat_dongs
        ),
        [
            "행정동명",
            "coherent_month_count",
            "disabled_total",
            "disabled_per_1000"
        ]
    ]
    .merge(
        disability_percentile[
            [
                "행정동명",
                "disabled_total__pct",
                "disabled_per_1000__pct"
            ]
        ],
        on="행정동명",
        how="left"
    )
    .sort_values(
        "disabled_per_1000",
        ascending=False
    )
    .round(3)
    .to_string(index=False)
)

=== 1. 반복지역 장애 Context 실제값 / Percentile ===

[신사동]
               feature    value  percentile
        disabled_total 356.0000     18.1818
     disabled_per_1000  23.2543     68.1818
   disabled_male_ratio   0.5955     45.4545
 disabled_female_ratio   0.4045     59.0909
    brain_lesion_ratio   0.1152     68.1818
          visual_ratio   0.0899     18.1818
          kidney_ratio   0.0702     59.0909
          autism_ratio   0.0421     36.3636
          mental_ratio   0.0169     13.6364
    intellectual_ratio   0.0421      9.0909
        physical_ratio   0.3961     86.3636
         hearing_ratio   0.1742     86.3636
other_disability_ratio   0.0534     95.4545

[일원본동]
               feature    value  percentile
        disabled_total 688.0000     72.7273
     disabled_per_1000  30.6712     81.8182
   disabled_male_ratio   0.4797      4.5455
 disabled_female_ratio   0.5203    100.0000
    brain_lesion_ratio   0.0858      9.0909
          visual_ratio   0.0727      4.5455
          kidney_

In [23]:
# =====================================================
# 1. 최종 프로파일에 사용할 대표 Context 변수
# =====================================================

profile_context_cols = [

    # 주민인구
    "resident_total_mean",
    "foreigner_ratio_mean",
    "resident_age_20_ratio",
    "resident_age_30_ratio",
    "resident_age_40_ratio",
    "resident_age_50_ratio",
    "resident_age_60_plus_ratio",

    # 1인가구
    "single_age_20_gap",
    "single_age_30_gap",
    "single_age_40_gap",
    "single_age_50_gap",
    "single_age_60_plus_gap",
    "official_single_household_mean",
    "official_single_household_change_rate",

    # 복지·경제 공통축
    "welfare_context_pc1",

    # 장애 Context
    "disabled_total",
    "disabled_per_1000"
]


# =====================================================
# 2. 행동신호 요약 변수
# =====================================================

behavior_summary_cols = [
    "coherent_month_count",
    "avg_low_weekday_outing_signal",
    "avg_very_low_outing_signal",
    "avg_outside_activity_axis",
    "avg_home_stay_axis",
    "max_core_signal_count"
]


# =====================================================
# 3. 기본 프로파일 생성
# =====================================================

region_profile = context_mart[
    [
        "행정동코드",
        "행정동명"
    ]
    + behavior_summary_cols
    + profile_context_cols
].copy()


# =====================================================
# 4. 기술적 반복 플래그
#
# 위험/고립 라벨이 아님
# 단지 동일 기준의 행동변화가
# 2개월 이상 관찰되었는지를 표시
# =====================================================

region_profile[
    "repeated_behavior_pattern"
] = (
    region_profile[
        "coherent_month_count"
    ] >= 2
).astype(int)


# =====================================================
# 5. Context percentile 추가
#
# 강남구 22개 동 내 상대적 위치
# =====================================================

for col in profile_context_cols:

    region_profile[
        f"{col}_pct"
    ] = (
        region_profile[col]
        .rank(
            pct=True,
            method="average",
            na_option="keep"
        )
        * 100
    )


# =====================================================
# 6. 검증
# =====================================================

print("=== 최종 지역 프로파일 검증 ===")

print("shape:", region_profile.shape)

print(
    "행정동 수:",
    region_profile[
        "행정동코드"
    ].nunique()
)

print(
    "중복:",
    region_profile.duplicated(
        ["행정동코드"]
    ).sum()
)

print(
    "전체 결측:",
    region_profile
    .isna()
    .sum()
    .sum()
)


# =====================================================
# 7. 행동신호 반복횟수 분포
# =====================================================

print(
    "\n=== coherent_month_count 분포 ==="
)

print(
    region_profile[
        "coherent_month_count"
    ]
    .value_counts()
    .sort_index()
)


# =====================================================
# 8. 반복 행동패턴 지역 프로파일 확인
# =====================================================

display_cols = [
    "행정동명",
    "coherent_month_count",

    "avg_low_weekday_outing_signal",
    "avg_very_low_outing_signal",
    "avg_outside_activity_axis",
    "avg_home_stay_axis",

    "resident_total_mean_pct",
    "resident_age_60_plus_ratio_pct",
    "single_age_60_plus_gap_pct",
    "welfare_context_pc1_pct",
    "disabled_per_1000_pct"
]


print(
    "\n=== 반복 행동패턴 지역 최종 요약 ==="
)

print(
    region_profile.loc[
        region_profile[
            "repeated_behavior_pattern"
        ] == 1,
        display_cols
    ]
    .sort_values(
        [
            "coherent_month_count",
            "행정동명"
        ],
        ascending=[
            False,
            True
        ]
    )
    .round(2)
    .to_string(index=False)
)


# =====================================================
# 9. 전체 22개 동 간단 요약
# =====================================================

print(
    "\n=== 전체 22개 동 행동패턴 현황 ==="
)

print(
    region_profile[
        [
            "행정동명",
            "coherent_month_count",
            "max_core_signal_count",
            "repeated_behavior_pattern"
        ]
    ]
    .sort_values(
        [
            "coherent_month_count",
            "max_core_signal_count"
        ],
        ascending=False
    )
    .to_string(index=False)
)

=== 최종 지역 프로파일 검증 ===
shape: (22, 43)
행정동 수: 22
중복: 0
전체 결측: 0

=== coherent_month_count 분포 ===
coherent_month_count
0    13
1     5
2     2
3     2
Name: count, dtype: int64

=== 반복 행동패턴 지역 최종 요약 ===
행정동명  coherent_month_count  avg_low_weekday_outing_signal  avg_very_low_outing_signal  avg_outside_activity_axis  avg_home_stay_axis  resident_total_mean_pct  resident_age_60_plus_ratio_pct  single_age_60_plus_gap_pct  welfare_context_pc1_pct  disabled_per_1000_pct
개포3동                     3                          -0.24                        1.79                       0.84                1.38                    22.73                           90.91                       95.45                    90.91                  90.91
일원본동                     3                           1.14                       -0.08                       0.24                0.54                    45.45                           50.00                      100.00                    27.27                  81.82
 

In [24]:
# =====================================================
# 1. 동별 coherent 발생 월 추출
# =====================================================

coherent_months = (
    outing_core.loc[
        outing_core["core_coherent_month"] == 1,
        ["행정동코드", "행정동명", "기준연월"]
    ]
    .groupby(
        ["행정동코드", "행정동명"]
    )["기준연월"]
    .apply(
        lambda x: ", ".join(
            sorted(
                x.astype(str).tolist()
            )
        )
    )
    .reset_index(
        name="coherent_months"
    )
)


# =====================================================
# 2. 최종 프로파일과 결합
# =====================================================

region_profile_final = (
    region_profile
    .merge(
        coherent_months,
        on=[
            "행정동코드",
            "행정동명"
        ],
        how="left",
        validate="one_to_one"
    )
)


# coherent 발생이 없는 동
region_profile_final[
    "coherent_months"
] = (
    region_profile_final[
        "coherent_months"
    ]
    .fillna("")
)


# =====================================================
# 3. FACT용 컬럼 정의
# =====================================================

fact_cols = [
    "행정동코드",
    "행정동명",

    "coherent_month_count",
    "coherent_months",

    "max_core_signal_count",

    "avg_low_weekday_outing_signal",
    "avg_very_low_outing_signal",
    "avg_outside_activity_axis",
    "avg_home_stay_axis",

    "repeated_behavior_pattern"
]


region_fact = (
    region_profile_final[
        fact_cols
    ]
    .copy()
)


# =====================================================
# 4. 검증
# =====================================================

print("=== FACT Table 검증 ===")
print("shape:", region_fact.shape)
print(
    "행정동 수:",
    region_fact["행정동코드"].nunique()
)
print(
    "중복:",
    region_fact.duplicated(
        ["행정동코드"]
    ).sum()
)


# =====================================================
# 5. 전체 22개 동 FACT 확인
# =====================================================

print(
    "\n=== 22개 동 행동신호 FACT ==="
)

print(
    region_fact[
        [
            "행정동명",
            "coherent_month_count",
            "coherent_months",
            "max_core_signal_count",
            "repeated_behavior_pattern"
        ]
    ]
    .sort_values(
        [
            "coherent_month_count",
            "max_core_signal_count"
        ],
        ascending=False
    )
    .to_string(index=False)
)


# =====================================================
# 6. 월 목록과 count 일치 여부 검증
# =====================================================

def count_months(x):

    if x == "":
        return 0

    return len(
        [
            v for v in x.split(", ")
            if v.strip() != ""
        ]
    )


region_fact[
    "month_list_count_check"
] = (
    region_fact[
        "coherent_months"
    ]
    .apply(count_months)
)


check = (
    region_fact[
        "month_list_count_check"
    ]
    ==
    region_fact[
        "coherent_month_count"
    ]
)


print(
    "\n=== 월 목록/count 일치 검증 ==="
)

print(
    "전체 일치:",
    check.all()
)

print(
    "불일치 행 수:",
    (~check).sum()
)

=== FACT Table 검증 ===
shape: (22, 10)
행정동 수: 22
중복: 0

=== 22개 동 행동신호 FACT ===
행정동명  coherent_month_count           coherent_months  max_core_signal_count  repeated_behavior_pattern
일원본동                     3 2025-10, 2025-11, 2025-12                      3                          1
개포3동                     3 2025-08, 2025-10, 2025-11                      3                          1
 신사동                     2          2025-08, 2025-09                      4                          1
일원1동                     2          2025-11, 2025-12                      4                          1
대치4동                     1                   2025-10                      3                          0
도곡1동                     1                   2025-12                      3                          0
개포4동                     1                   2025-09                      3                          0
대치2동                     1                   2025-08                      2                      

In [25]:
# =====================================================
# 1. CONTEXT 변수
# =====================================================

context_service_cols = [

    # 주민인구
    "resident_total_mean",
    "foreigner_ratio_mean",
    "resident_age_20_ratio",
    "resident_age_30_ratio",
    "resident_age_40_ratio",
    "resident_age_50_ratio",
    "resident_age_60_plus_ratio",

    # 1인가구
    "single_age_20_gap",
    "single_age_30_gap",
    "single_age_40_gap",
    "single_age_50_gap",
    "single_age_60_plus_gap",
    "official_single_household_mean",
    "official_single_household_change_rate",

    # 복지·경제
    "welfare_context_pc1",

    # 장애
    "disabled_total",
    "disabled_per_1000"
]


# =====================================================
# 2. CONTEXT Table 생성
# =====================================================

context_output_cols = [
    "행정동코드",
    "행정동명"
]

for col in context_service_cols:
    context_output_cols.append(col)
    context_output_cols.append(f"{col}_pct")


region_context = (
    region_profile_final[
        context_output_cols
    ]
    .copy()
)


# =====================================================
# 3. 검증
# =====================================================

print("=== CONTEXT Table 검증 ===")
print("shape:", region_context.shape)

print(
    "행정동 수:",
    region_context["행정동코드"].nunique()
)

print(
    "중복:",
    region_context.duplicated(
        ["행정동코드"]
    ).sum()
)

print(
    "전체 결측:",
    region_context.isna().sum().sum()
)


# =====================================================
# 4. 반복지역 핵심 Context 확인
# =====================================================

context_check_cols = [
    "행정동명",

    "resident_total_mean_pct",
    "resident_age_60_plus_ratio_pct",

    "single_age_20_gap_pct",
    "single_age_30_gap_pct",
    "single_age_60_plus_gap_pct",

    "welfare_context_pc1_pct",

    "disabled_per_1000_pct"
]


print(
    "\n=== 반복 행동패턴 4개 동 Context ==="
)

print(
    region_context.loc[
        region_context["행정동명"].isin(
            repeat_dongs
        ),
        context_check_cols
    ]
    .round(1)
    .to_string(index=False)
)


# =====================================================
# 5. percentile 범위 검증
# =====================================================

pct_cols = [
    col for col in region_context.columns
    if col.endswith("_pct")
]

print(
    "\n=== Percentile 범위 검증 ==="
)

print(
    "Percentile 변수 수:",
    len(pct_cols)
)

print(
    "최솟값:",
    region_context[pct_cols]
    .min()
    .min()
)

print(
    "최댓값:",
    region_context[pct_cols]
    .max()
    .max()
)

print(
    "범위 오류 수:",
    (
        (
            region_context[pct_cols] < 0
        )
        |
        (
            region_context[pct_cols] > 100
        )
    )
    .sum()
    .sum()
)

=== CONTEXT Table 검증 ===
shape: (22, 36)
행정동 수: 22
중복: 0
전체 결측: 0

=== 반복 행동패턴 4개 동 Context ===
행정동명  resident_total_mean_pct  resident_age_60_plus_ratio_pct  single_age_20_gap_pct  single_age_30_gap_pct  single_age_60_plus_gap_pct  welfare_context_pc1_pct  disabled_per_1000_pct
 신사동                     18.2                            77.3                   90.9                   81.8                        31.8                     59.1                   68.2
일원본동                     45.5                            50.0                   27.3                    9.1                       100.0                     27.3                   81.8
일원1동                     13.6                            95.5                   63.6                   22.7                        63.6                     95.5                   95.5
개포3동                     22.7                            90.9                   13.6                   13.6                        95.5                     90.9        

In [26]:
import pandas as pd
import numpy as np


# =====================================================
# 1. Context 상대위치 태그 함수
#
# 위험등급이 아니라 강남구 22개 동 내
# 상대적 위치를 설명하기 위한 표현
# =====================================================

def context_position(pct):

    if pd.isna(pct):
        return "정보없음"

    elif pct >= 80:
        return "상대적으로 높음"

    elif pct < 20:
        return "상대적으로 낮음"

    else:
        return "중간 범위"


# =====================================================
# 2. 해석에 사용할 핵심 Context
#
# 모든 17개를 태그화하지 않고
# 지금까지 실제 해석에서 의미가 있었던
# 대표 변수 중심
# =====================================================

interpret_context = {
    "resident_total_mean_pct":
        "주민등록인구 규모",

    "resident_age_60_plus_ratio_pct":
        "60세 이상 주민 비율",

    "single_age_20_gap_pct":
        "20대 1인가구 상대집중",

    "single_age_30_gap_pct":
        "30대 1인가구 상대집중",

    "single_age_60_plus_gap_pct":
        "60세 이상 1인가구 상대집중",

    "welfare_context_pc1_pct":
        "복지·경제 관련 공통축",

    "disabled_per_1000_pct":
        "인구 대비 등록장애인 관련 지표"
}


# =====================================================
# 3. INTERPRETATION 기본 테이블
# =====================================================

region_interpretation = region_fact[
    [
        "행정동코드",
        "행정동명",
        "coherent_month_count",
        "coherent_months",
        "max_core_signal_count"
    ]
].copy()


# Context 결합
region_interpretation = (
    region_interpretation
    .merge(
        region_context[
            [
                "행정동코드",
                "행정동명"
            ]
            + list(
                interpret_context.keys()
            )
        ],
        on=[
            "행정동코드",
            "행정동명"
        ],
        how="left",
        validate="one_to_one"
    )
)


# =====================================================
# 4. Context 설명 태그 생성
# =====================================================

for pct_col, label in interpret_context.items():

    tag_col = (
        pct_col
        .replace("_pct", "")
        + "_position"
    )

    region_interpretation[
        tag_col
    ] = (
        region_interpretation[
            pct_col
        ]
        .apply(context_position)
    )


# =====================================================
# 5. 행동신호 설명용 상태
#
# 위험등급이 아님
# 관찰된 월 수만 기술적으로 표현
# =====================================================

def behavior_observation_text(row):

    count = row[
        "coherent_month_count"
    ]

    months = row[
        "coherent_months"
    ]

    if count == 0:
        return (
            "분석기간 중 외출 관련 "
            "핵심 행동신호의 월 단위 "
            "동시 관찰 없음"
        )

    elif count == 1:
        return (
            f"{months}에 외출 관련 "
            "핵심 행동신호가 동시 관찰됨"
        )

    else:
        return (
            f"분석기간 중 {count}개 월"
            f"({months})에서 외출 관련 "
            "핵심 행동신호가 동시 관찰됨"
        )


region_interpretation[
    "behavior_observation"
] = (
    region_interpretation
    .apply(
        behavior_observation_text,
        axis=1
    )
)


# =====================================================
# 6. 검증
# =====================================================

print(
    "=== INTERPRETATION Table 검증 ==="
)

print(
    "shape:",
    region_interpretation.shape
)

print(
    "행정동 수:",
    region_interpretation[
        "행정동코드"
    ].nunique()
)

print(
    "중복:",
    region_interpretation
    .duplicated(
        ["행정동코드"]
    )
    .sum()
)

print(
    "결측:",
    region_interpretation
    .isna()
    .sum()
    .sum()
)


# =====================================================
# 7. 반복 행동패턴 4개 동 확인
# =====================================================

position_cols = [
    col
    for col in region_interpretation.columns
    if col.endswith("_position")
]


print(
    "\n=== 반복 4개 동 해석용 데이터 ==="
)

print(
    region_interpretation.loc[
        region_interpretation[
            "coherent_month_count"
        ] >= 2,
        [
            "행정동명",
            "behavior_observation"
        ]
        + position_cols
    ]
    .to_string(index=False)
)

=== INTERPRETATION Table 검증 ===
shape: (22, 20)
행정동 수: 22
중복: 0
결측: 0

=== 반복 4개 동 해석용 데이터 ===
행정동명                                           behavior_observation resident_total_mean_position resident_age_60_plus_ratio_position single_age_20_gap_position single_age_30_gap_position single_age_60_plus_gap_position welfare_context_pc1_position disabled_per_1000_position
 신사동          분석기간 중 2개 월(2025-08, 2025-09)에서 외출 관련 핵심 행동신호가 동시 관찰됨                     상대적으로 낮음                               중간 범위                   상대적으로 높음                   상대적으로 높음                           중간 범위                        중간 범위                      중간 범위
일원본동 분석기간 중 3개 월(2025-10, 2025-11, 2025-12)에서 외출 관련 핵심 행동신호가 동시 관찰됨                        중간 범위                               중간 범위                      중간 범위                   상대적으로 낮음                        상대적으로 높음                        중간 범위                   상대적으로 높음
일원1동          분석기간 중 2개 월(2025-11, 2025-12)에서 외출 관련 핵심 행동신호가 동시 관찰됨             

In [27]:
import pandas as pd
from google.colab import files


# =====================================================
# 1. 최종 저장 전 불필요한 검증용 컬럼 제거
# =====================================================

region_fact_final = region_fact.drop(
    columns=["month_list_count_check"],
    errors="ignore"
).copy()


region_context_final = (
    region_context.copy()
)

region_interpretation_final = (
    region_interpretation.copy()
)


# =====================================================
# 2. 최종 검증
# =====================================================

tables = {
    "FACT": region_fact_final,
    "CONTEXT": region_context_final,
    "INTERPRETATION": region_interpretation_final
}


print("=== 최종 테이블 검증 ===")

for name, df in tables.items():

    print(
        f"{name}: "
        f"shape={df.shape}, "
        f"동={df['행정동코드'].nunique()}, "
        f"중복={df.duplicated(['행정동코드']).sum()}, "
        f"결측={df.isna().sum().sum()}"
    )


# =====================================================
# 3. CSV 저장
# =====================================================

fact_filename = (
    "gangnam_db1_region_behavior_fact_2025.csv"
)

context_filename = (
    "gangnam_db1_region_context_2025.csv"
)

interpret_filename = (
    "gangnam_db1_region_interpretation_2025.csv"
)


region_fact_final.to_csv(
    fact_filename,
    index=False,
    encoding="utf-8-sig"
)

region_context_final.to_csv(
    context_filename,
    index=False,
    encoding="utf-8-sig"
)

region_interpretation_final.to_csv(
    interpret_filename,
    index=False,
    encoding="utf-8-sig"
)


print("\n=== 저장 완료 ===")
print(fact_filename)
print(context_filename)
print(interpret_filename)

=== 최종 테이블 검증 ===
FACT: shape=(22, 10), 동=22, 중복=0, 결측=0
CONTEXT: shape=(22, 36), 동=22, 중복=0, 결측=0
INTERPRETATION: shape=(22, 20), 동=22, 중복=0, 결측=0

=== 저장 완료 ===
gangnam_db1_region_behavior_fact_2025.csv
gangnam_db1_region_context_2025.csv
gangnam_db1_region_interpretation_2025.csv


In [28]:
from google.colab import files

files.download(
    "gangnam_db1_region_behavior_fact_2025.csv"
)

files.download(
    "gangnam_db1_region_context_2025.csv"
)

files.download(
    "gangnam_db1_region_interpretation_2025.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>